<a href="https://colab.research.google.com/github/Al-Amin95/PromptInjectionDetectionSystem/blob/model-train-comparison/notebooks/03_distilbert_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#DistilBERT Fine Tuning

# Part 1 - Fine-Tuning Concept and Objective

## 1.1 Fine-Tuning

* Fine-tuning means training a pre-trained model for a specific task.
* Here, the task is prompt injection detection.
* The model classifies prompts as:

  * SAFE
  * INJECTION

## 1.2 Why DistilBERT

* DistilBERT is used as the baseline model.
* Its results will be compared later with:

  * RoBERTa
  * SecBERT

## 1.3 Task Type

* This is binary text classification.
* Labels:

  * `0 = SAFE`
  * `1 = INJECTION`

## 1.4 Dataset

* Dataset: prepared deepset prompt-injection dataset.
* Files:

  * `clean_train.parquet`
  * `clean_validation.parquet`
  * `clean_test.parquet`
* Input column: `text_for_model`
* Label column: `label`

## 1.5 Objective

* Fine-tune DistilBERT to detect prompt injection.
* Evaluate the model on validation and test data.
* Save results for comparison with other models.

## 1.6 Expected Outputs

* Fine-tuned DistilBERT model
* Saved tokenizer
* Validation and test metrics
* Confusion matrix
* Prediction files
* False positive and false negative examples
* Training summary


# Part 2 - Notebook Setup and Repository Update

In [ ]:
# 1 mount google drive
from google.colab import drive
drive.mount("/content/drive")

print("drive ready")


# 2 import path
from pathlib import Path


# 3 config
repo_url = "https://github.com/Al-Amin95/PromptInjectionDetectionSystem.git"
branch = "model-train-comparison"
repo_dir = Path("/content/PromptInjectionDetectionSystem")

drive_base = Path("/content/drive/MyDrive/USW_Dissertation/Prompt-Injection-Detection-System-SHAP")

print("repo:", repo_url)
print("branch:", branch)
print("drive base:", drive_base)


# 4 clone or update repo
if repo_dir.exists():
    print("repo exists, updating")
    %cd /content/PromptInjectionDetectionSystem
    !git fetch origin
    !git checkout {branch}
    !git pull origin {branch}
else:
    print("cloning repo")
    %cd /content
    !git clone -b {branch} {repo_url}
    %cd /content/PromptInjectionDetectionSystem

print("repo ready")


# 5 install libraries
print("installing libraries")

requirements_path = Path("/content/PromptInjectionDetectionSystem/requirements.txt")

if requirements_path.exists():
    !pip install -q -r requirements.txt
else:
    print("requirements.txt not found, installing main libraries only")

!pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy matplotlib pyarrow

print("libraries ready")


# 6 import packages
import os
import sys
import time
import json
import shutil
import random
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import transformers
import datasets
import sklearn

print("packages imported")


# 7 define repo paths
project_root = Path("/content/PromptInjectionDetectionSystem")

raw_data_dir = project_root / "data" / "raw"
processed_data_dir = project_root / "data" / "processed"
notebooks_dir = project_root / "notebooks"

results_dir = project_root / "results"
results_tables_dir = results_dir / "tables"
results_figures_dir = results_dir / "figures"
results_predictions_dir = results_dir / "predictions"
results_errors_dir = results_dir / "error_analysis"

train_path = processed_data_dir / "clean_train.parquet"
validation_path = processed_data_dir / "clean_validation.parquet"
test_path = processed_data_dir / "clean_test.parquet"

print("repo paths ready")


# 8 define drive paths
drive_data_dir = drive_base / "data"
drive_models_dir = drive_base / "models"
drive_notebooks_dir = drive_base / "notebooks"
drive_results_dir = drive_base / "results"
drive_screenshots_dir = drive_base / "screenshots"
drive_shap_outputs_dir = drive_base / "shap_outputs"
drive_dissertation_evidence_dir = drive_base / "dissertation_evidence"
drive_github_backup_dir = drive_base / "github_repo_backup"

drive_distilbert_dir = drive_models_dir / "distilbert"
drive_roberta_dir = drive_models_dir / "roberta"
drive_secbert_dir = drive_models_dir / "secbert"

drive_results_tables_dir = drive_results_dir / "tables"
drive_results_figures_dir = drive_results_dir / "figures"
drive_results_predictions_dir = drive_results_dir / "predictions"
drive_results_errors_dir = drive_results_dir / "error_analysis"

print("drive paths ready")


# 9 create output folders
folders_to_create = [
    results_dir,
    results_tables_dir,
    results_figures_dir,
    results_predictions_dir,
    results_errors_dir,
    drive_data_dir,
    drive_models_dir,
    drive_notebooks_dir,
    drive_results_dir,
    drive_screenshots_dir,
    drive_shap_outputs_dir,
    drive_dissertation_evidence_dir,
    drive_github_backup_dir,
    drive_distilbert_dir,
    drive_roberta_dir,
    drive_secbert_dir,
    drive_results_tables_dir,
    drive_results_figures_dir,
    drive_results_predictions_dir,
    drive_results_errors_dir,
]

for folder in folders_to_create:
    folder.mkdir(parents=True, exist_ok=True)

print("folders ready")


# 10 check dataset files
print("dataset check")
print("train:", train_path.exists(), "|", train_path)
print("validation:", validation_path.exists(), "|", validation_path)
print("test:", test_path.exists(), "|", test_path)


# 11 check drive model folders
print("drive model folder check")
print("distilbert:", drive_distilbert_dir.exists(), "|", drive_distilbert_dir)
print("roberta:", drive_roberta_dir.exists(), "|", drive_roberta_dir)
print("secbert:", drive_secbert_dir.exists(), "|", drive_secbert_dir)


# 12 check branch
current_branch = !git branch --show-current
current_branch = current_branch[0].strip()

print("branch check")
print("expected:", branch)
print("current:", current_branch)


# 13 check environment
try:
    import google.colab
    running_in_colab = True
except ImportError:
    running_in_colab = False

print("environment check")
print("running in colab:", running_in_colab)
print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("sklearn:", sklearn.__version__)


# 14 final setup check
all_datasets_exist = train_path.exists() and validation_path.exists() and test_path.exists()
all_model_folders_exist = drive_distilbert_dir.exists() and drive_roberta_dir.exists() and drive_secbert_dir.exists()
correct_branch = current_branch == branch

print("final check")
print("correct branch:", correct_branch)
print("all prepared datasets exist:", all_datasets_exist)
print("all drive model folders exist:", all_model_folders_exist)
print("running in colab:", running_in_colab)

if correct_branch and all_datasets_exist and all_model_folders_exist and running_in_colab:
    print("part 2 complete")
else:
    print("part 2 needs checking")

#Part 3 - Define Paths and Output Folders

In [ ]:
# 1 define experiment name
model_name = "distilbert"
model_checkpoint = "distilbert-base-uncased"
experiment_name = "distilbert_baseline"

print("model name:", model_name)
print("model checkpoint:", model_checkpoint)
print("experiment name:", experiment_name)


# 2 confirm base paths from part 2
project_root = Path("/content/PromptInjectionDetectionSystem")
drive_base = Path("/content/drive/MyDrive/USW_Dissertation/Prompt-Injection-Detection-System-SHAP")

print("project root:", project_root)
print("drive base:", drive_base)


# 3 define processed dataset paths
processed_data_dir = project_root / "data" / "processed"

train_path = processed_data_dir / "clean_train.parquet"
validation_path = processed_data_dir / "clean_validation.parquet"
test_path = processed_data_dir / "clean_test.parquet"

print("train path:", train_path)
print("validation path:", validation_path)
print("test path:", test_path)


# 4 define repo output folders
repo_results_dir = project_root / "results"

repo_tables_dir = repo_results_dir / "tables" / model_name
repo_figures_dir = repo_results_dir / "figures" / model_name
repo_predictions_dir = repo_results_dir / "predictions" / model_name
repo_errors_dir = repo_results_dir / "error_analysis" / model_name
repo_logs_dir = repo_results_dir / "logs" / model_name

repo_notebooks_dir = project_root / "notebooks"

print("repo output folders defined")


# 5 define drive output folders
drive_models_dir = drive_base / "models"
drive_model_dir = drive_models_dir / model_name

drive_results_dir = drive_base / "results"
drive_tables_dir = drive_results_dir / "tables" / model_name
drive_figures_dir = drive_results_dir / "figures" / model_name
drive_predictions_dir = drive_results_dir / "predictions" / model_name
drive_errors_dir = drive_results_dir / "error_analysis" / model_name
drive_logs_dir = drive_results_dir / "logs" / model_name

drive_notebooks_dir = drive_base / "notebooks"
drive_screenshots_dir = drive_base / "screenshots"
drive_evidence_dir = drive_base / "dissertation_evidence" / model_name

print("drive output folders defined")


# 6 define temporary training folder
# this folder is outside github repo to avoid committing large checkpoint files
training_output_dir = Path("/content/distilbert_training_output")

print("training output dir:", training_output_dir)


# 7 create all output folders
folders_to_create = [
    repo_tables_dir,
    repo_figures_dir,
    repo_predictions_dir,
    repo_errors_dir,
    repo_logs_dir,
    repo_notebooks_dir,
    drive_model_dir,
    drive_tables_dir,
    drive_figures_dir,
    drive_predictions_dir,
    drive_errors_dir,
    drive_logs_dir,
    drive_notebooks_dir,
    drive_screenshots_dir,
    drive_evidence_dir,
    training_output_dir,
]

for folder in folders_to_create:
    folder.mkdir(parents=True, exist_ok=True)

print("folders created")


# 8 check dataset files
dataset_paths = {
    "train": train_path,
    "validation": validation_path,
    "test": test_path,
}

print("dataset file check")
for name, path in dataset_paths.items():
    print(name, "=", path.exists(), "|", path)


# 9 check repo output folders
repo_output_folders = {
    "repo tables": repo_tables_dir,
    "repo figures": repo_figures_dir,
    "repo predictions": repo_predictions_dir,
    "repo error analysis": repo_errors_dir,
    "repo logs": repo_logs_dir,
    "repo notebooks": repo_notebooks_dir,
}

print("repo folder check")
for name, path in repo_output_folders.items():
    print(name, "=", path.exists(), "|", path)


# 10 check drive output folders
drive_output_folders = {
    "drive model": drive_model_dir,
    "drive tables": drive_tables_dir,
    "drive figures": drive_figures_dir,
    "drive predictions": drive_predictions_dir,
    "drive error analysis": drive_errors_dir,
    "drive logs": drive_logs_dir,
    "drive notebooks": drive_notebooks_dir,
    "drive screenshots": drive_screenshots_dir,
    "drive evidence": drive_evidence_dir,
}

print("drive folder check")
for name, path in drive_output_folders.items():
    print(name, "=", path.exists(), "|", path)


# 11 check notebook folder typo
wrong_notebook_dir = project_root / "notebiooks"
correct_notebook_dir = project_root / "notebooks"

print("notebook folder check")
print("correct notebooks folder exists:", correct_notebook_dir.exists(), "|", correct_notebook_dir)
print("wrong notebiooks folder exists:", wrong_notebook_dir.exists(), "|", wrong_notebook_dir)


# 12 define common output file paths
validation_metrics_path = repo_tables_dir / "distilbert_validation_metrics.csv"
test_metrics_path = repo_tables_dir / "distilbert_test_metrics.csv"
training_summary_path = repo_tables_dir / "distilbert_training_summary.csv"

validation_predictions_path = repo_predictions_dir / "distilbert_validation_predictions.csv"
test_predictions_path = repo_predictions_dir / "distilbert_test_predictions.csv"

confusion_matrix_table_path = repo_tables_dir / "distilbert_confusion_matrix.csv"
confusion_matrix_figure_path = repo_figures_dir / "distilbert_confusion_matrix.png"

false_positive_path = repo_errors_dir / "distilbert_false_positives.csv"
false_negative_path = repo_errors_dir / "distilbert_false_negatives.csv"

print("main output file paths ready")


# 13 final part 3 check
all_datasets_exist = all(path.exists() for path in dataset_paths.values())
all_repo_folders_exist = all(path.exists() for path in repo_output_folders.values())
all_drive_folders_exist = all(path.exists() for path in drive_output_folders.values())
training_folder_exists = training_output_dir.exists()

print("final check")
print("all datasets exist:", all_datasets_exist)
print("all repo output folders exist:", all_repo_folders_exist)
print("all drive output folders exist:", all_drive_folders_exist)
print("training folder exists:", training_folder_exists)

if all_datasets_exist and all_repo_folders_exist and all_drive_folders_exist and training_folder_exists:
    print("part 3 complete")
else:
    print("part 3 needs checking")

#Part 4 - Check GPU Availability

In [ ]:
# 1 check cuda and gpu
cuda_available = torch.cuda.is_available()

print("cuda available:", cuda_available)


# 2 get device
if cuda_available:
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_count = torch.cuda.device_count()
else:
    device = torch.device("cpu")
    gpu_name = "no gpu available"
    gpu_count = 0

print("device:", device)
print("gpu name:", gpu_name)
print("gpu count:", gpu_count)


# 3 check gpu memory if available
if cuda_available:
    total_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    allocated_memory_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
    reserved_memory_gb = torch.cuda.memory_reserved(0) / (1024 ** 3)

    print("total gpu memory gb:", round(total_memory_gb, 2))
    print("allocated gpu memory gb:", round(allocated_memory_gb, 2))
    print("reserved gpu memory gb:", round(reserved_memory_gb, 2))
else:
    print("gpu memory check skipped because gpu is not available")


# 4 save device information
device_info = {
    "cuda_available": cuda_available,
    "device": str(device),
    "gpu_name": gpu_name,
    "gpu_count": gpu_count,
}

if cuda_available:
    device_info["total_gpu_memory_gb"] = round(total_memory_gb, 2)
    device_info["allocated_gpu_memory_gb"] = round(allocated_memory_gb, 2)
    device_info["reserved_gpu_memory_gb"] = round(reserved_memory_gb, 2)

device_info_path = repo_logs_dir / "distilbert_device_info.json"
drive_device_info_path = drive_logs_dir / "distilbert_device_info.json"

with open(device_info_path, "w") as f:
    json.dump(device_info, f, indent=4)

with open(drive_device_info_path, "w") as f:
    json.dump(device_info, f, indent=4)

print("device info saved:", device_info_path)
print("device info backup saved:", drive_device_info_path)


# 5 final gpu check
print("final gpu check")

if cuda_available:
    print("part 4 complete: gpu is available")
else:
    print("part 4 warning: gpu is not available")
    print("go to runtime > change runtime type > hardware accelerator > gpu")

#Part 5 - Load Prepared Train, Validation, and Test Datasets

In [ ]:
# 1 load prepared datasets
train_df = pd.read_parquet(train_path)
validation_df = pd.read_parquet(validation_path)
test_df = pd.read_parquet(test_path)

print("datasets loaded")


# 2 check dataset shapes
print("dataset shapes")
print("train:", train_df.shape)
print("validation:", validation_df.shape)
print("test:", test_df.shape)


# 3 check columns
print("columns")
print("train columns:", train_df.columns.tolist())
print("validation columns:", validation_df.columns.tolist())
print("test columns:", test_df.columns.tolist())


# 4 preview train data
print("train preview")
display(train_df.head())


# 5 preview validation data
print("validation preview")
display(validation_df.head())


# 6 preview test data
print("test preview")
display(test_df.head())


# 7 check label counts
print("label counts")
print("train labels")
print(train_df["label"].value_counts().sort_index())

print("validation labels")
print(validation_df["label"].value_counts().sort_index())

print("test labels")
print(test_df["label"].value_counts().sort_index())


# 8 check split counts if split column exists
print("split counts")

if "split" in train_df.columns:
    print("train split")
    print(train_df["split"].value_counts())

if "split" in validation_df.columns:
    print("validation split")
    print(validation_df["split"].value_counts())

if "split" in test_df.columns:
    print("test split")
    print(test_df["split"].value_counts())


# 9 check expected row counts
expected_train_rows = 436
expected_validation_rows = 110
expected_test_rows = 116

train_rows_ok = len(train_df) == expected_train_rows
validation_rows_ok = len(validation_df) == expected_validation_rows
test_rows_ok = len(test_df) == expected_test_rows

print("row count check")
print("train rows correct:", train_rows_ok)
print("validation rows correct:", validation_rows_ok)
print("test rows correct:", test_rows_ok)


# 10 final part 5 check
if train_rows_ok and validation_rows_ok and test_rows_ok:
    print("part 5 complete")
else:
    print("part 5 needs checking")

#Part 6 - Verify Dataset Columns and Split Integrity

In [ ]:
# 1 define required columns
required_columns = ["id", "original_text", "text_for_model", "label", "split"]

print("required columns:", required_columns)


# 2 check required columns
def check_required_columns(df, df_name):
    missing_columns = [col for col in required_columns if col not in df.columns]

    print(df_name, "missing columns:", missing_columns)

    if len(missing_columns) == 0:
        return True
    else:
        return False


train_columns_ok = check_required_columns(train_df, "train")
validation_columns_ok = check_required_columns(validation_df, "validation")
test_columns_ok = check_required_columns(test_df, "test")


# 3 check binary labels
def check_binary_labels(df, df_name):
    unique_labels = sorted(df["label"].dropna().unique().tolist())
    labels_ok = unique_labels == [0, 1]

    print(df_name, "unique labels:", unique_labels)
    print(df_name, "labels ok:", labels_ok)

    return labels_ok


train_labels_ok = check_binary_labels(train_df, "train")
validation_labels_ok = check_binary_labels(validation_df, "validation")
test_labels_ok = check_binary_labels(test_df, "test")


# 4 check split values
def check_split_value(df, df_name, expected_split):
    unique_splits = sorted(df["split"].dropna().unique().tolist())
    split_ok = unique_splits == [expected_split]

    print(df_name, "unique split values:", unique_splits)
    print(df_name, "split ok:", split_ok)

    return split_ok


train_split_ok = check_split_value(train_df, "train", "train")
validation_split_ok = check_split_value(validation_df, "validation", "validation")
test_split_ok = check_split_value(test_df, "test", "test")


# 5 check missing text_for_model
def check_missing_text(df, df_name):
    missing_text_count = df["text_for_model"].isna().sum()
    empty_text_count = (df["text_for_model"].astype(str).str.strip() == "").sum()

    text_ok = missing_text_count == 0 and empty_text_count == 0

    print(df_name, "missing text_for_model:", missing_text_count)
    print(df_name, "empty text_for_model:", empty_text_count)
    print(df_name, "text ok:", text_ok)

    return text_ok


train_text_ok = check_missing_text(train_df, "train")
validation_text_ok = check_missing_text(validation_df, "validation")
test_text_ok = check_missing_text(test_df, "test")


# 6 check missing labels
def check_missing_labels(df, df_name):
    missing_label_count = df["label"].isna().sum()
    label_ok = missing_label_count == 0

    print(df_name, "missing labels:", missing_label_count)
    print(df_name, "missing label ok:", label_ok)

    return label_ok


train_missing_label_ok = check_missing_labels(train_df, "train")
validation_missing_label_ok = check_missing_labels(validation_df, "validation")
test_missing_label_ok = check_missing_labels(test_df, "test")


# 7 check id overlap between splits
train_ids = set(train_df["id"].astype(str))
validation_ids = set(validation_df["id"].astype(str))
test_ids = set(test_df["id"].astype(str))

train_validation_overlap = train_ids.intersection(validation_ids)
train_test_overlap = train_ids.intersection(test_ids)
validation_test_overlap = validation_ids.intersection(test_ids)

print("id overlap check")
print("train-validation overlap:", len(train_validation_overlap))
print("train-test overlap:", len(train_test_overlap))
print("validation-test overlap:", len(validation_test_overlap))

id_overlap_ok = (
    len(train_validation_overlap) == 0 and
    len(train_test_overlap) == 0 and
    len(validation_test_overlap) == 0
)

print("id overlap ok:", id_overlap_ok)


# 8 check text overlap between splits
train_texts = set(train_df["text_for_model"].astype(str))
validation_texts = set(validation_df["text_for_model"].astype(str))
test_texts = set(test_df["text_for_model"].astype(str))

train_validation_text_overlap = train_texts.intersection(validation_texts)
train_test_text_overlap = train_texts.intersection(test_texts)
validation_test_text_overlap = validation_texts.intersection(test_texts)

print("text overlap check")
print("train-validation text overlap:", len(train_validation_text_overlap))
print("train-test text overlap:", len(train_test_text_overlap))
print("validation-test text overlap:", len(validation_test_text_overlap))

text_overlap_ok = (
    len(train_validation_text_overlap) == 0 and
    len(train_test_text_overlap) == 0 and
    len(validation_test_text_overlap) == 0
)

print("text overlap ok:", text_overlap_ok)


# 9 check label data type
print("label data types")
print("train label dtype:", train_df["label"].dtype)
print("validation label dtype:", validation_df["label"].dtype)
print("test label dtype:", test_df["label"].dtype)

label_dtype_ok = (
    pd.api.types.is_integer_dtype(train_df["label"]) and
    pd.api.types.is_integer_dtype(validation_df["label"]) and
    pd.api.types.is_integer_dtype(test_df["label"])
)

print("label dtype ok:", label_dtype_ok)


# 10 save dataset integrity summary
dataset_integrity_summary = pd.DataFrame([
    {"check": "train columns ok", "status": train_columns_ok},
    {"check": "validation columns ok", "status": validation_columns_ok},
    {"check": "test columns ok", "status": test_columns_ok},
    {"check": "train labels ok", "status": train_labels_ok},
    {"check": "validation labels ok", "status": validation_labels_ok},
    {"check": "test labels ok", "status": test_labels_ok},
    {"check": "train split ok", "status": train_split_ok},
    {"check": "validation split ok", "status": validation_split_ok},
    {"check": "test split ok", "status": test_split_ok},
    {"check": "train text ok", "status": train_text_ok},
    {"check": "validation text ok", "status": validation_text_ok},
    {"check": "test text ok", "status": test_text_ok},
    {"check": "train missing labels ok", "status": train_missing_label_ok},
    {"check": "validation missing labels ok", "status": validation_missing_label_ok},
    {"check": "test missing labels ok", "status": test_missing_label_ok},
    {"check": "id overlap ok", "status": id_overlap_ok},
    {"check": "text overlap ok", "status": text_overlap_ok},
    {"check": "label dtype ok", "status": label_dtype_ok},
])

dataset_integrity_path = repo_tables_dir / "distilbert_dataset_integrity_summary.csv"
drive_dataset_integrity_path = drive_tables_dir / "distilbert_dataset_integrity_summary.csv"

dataset_integrity_summary.to_csv(dataset_integrity_path, index=False)
dataset_integrity_summary.to_csv(drive_dataset_integrity_path, index=False)

print("dataset integrity summary saved:", dataset_integrity_path)
print("dataset integrity summary backup saved:", drive_dataset_integrity_path)


# 11 final part 6 check
all_integrity_checks_passed = dataset_integrity_summary["status"].all()

print("final check")
print("all dataset integrity checks passed:", all_integrity_checks_passed)

if all_integrity_checks_passed:
    print("part 6 complete")
else:
    print("part 6 needs checking")
    display(dataset_integrity_summary)

#Part 7 - Define DistilBERT Model Configuration

In [ ]:
# 1 define model configuration
model_name = "distilbert"
model_checkpoint = "distilbert-base-uncased"
experiment_name = "distilbert_baseline"

task_type = "binary text classification"
problem_type = "single_label_classification"
num_labels = 2

max_length = 512
random_seed = 42

print("model configuration ready")


# 2 define label mapping
label2id = {
    "SAFE": 0,
    "INJECTION": 1
}

id2label = {
    0: "SAFE",
    1: "INJECTION"
}

positive_label = 1
positive_label_name = "INJECTION"

print("label mapping ready")
print("label2id:", label2id)
print("id2label:", id2label)
print("positive label:", positive_label, positive_label_name)


# 3 define tokenizer settings
padding_strategy = "max_length"
truncation = True
use_fast_tokenizer = True

print("tokenizer settings ready")
print("max length:", max_length)
print("padding:", padding_strategy)
print("truncation:", truncation)


# 4 define training selection rule
main_selection_metric = "f1"
greater_is_better = True

print("model selection rule ready")
print("main metric:", main_selection_metric)
print("greater is better:", greater_is_better)


# 5 set random seed
from transformers import set_seed

random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
set_seed(random_seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)

print("random seed set:", random_seed)


# 6 define output names
model_run_name = "distilbert_prompt_injection_baseline"

local_best_model_folder = drive_model_dir / "best_model"
local_tokenizer_folder = drive_model_dir / "tokenizer"

repo_config_path = repo_logs_dir / "distilbert_model_config.json"
drive_config_path = drive_logs_dir / "distilbert_model_config.json"

repo_config_table_path = repo_tables_dir / "distilbert_model_config.csv"
drive_config_table_path = drive_tables_dir / "distilbert_model_config.csv"

print("output names ready")
print("model run name:", model_run_name)
print("best model folder:", local_best_model_folder)
print("tokenizer folder:", local_tokenizer_folder)


# 7 save configuration
distilbert_config = {
    "model_name": model_name,
    "model_checkpoint": model_checkpoint,
    "experiment_name": experiment_name,
    "model_run_name": model_run_name,
    "task_type": task_type,
    "problem_type": problem_type,
    "num_labels": num_labels,
    "max_length": max_length,
    "random_seed": random_seed,
    "label2id": label2id,
    "id2label": id2label,
    "positive_label": positive_label,
    "positive_label_name": positive_label_name,
    "padding_strategy": padding_strategy,
    "truncation": truncation,
    "use_fast_tokenizer": use_fast_tokenizer,
    "main_selection_metric": main_selection_metric,
    "greater_is_better": greater_is_better,
    "training_output_dir": str(training_output_dir),
    "drive_model_dir": str(drive_model_dir),
    "best_model_folder": str(local_best_model_folder),
    "tokenizer_folder": str(local_tokenizer_folder)
}

with open(repo_config_path, "w") as f:
    json.dump(distilbert_config, f, indent=4)

with open(drive_config_path, "w") as f:
    json.dump(distilbert_config, f, indent=4)

config_table = pd.DataFrame([
    {"setting": key, "value": str(value)}
    for key, value in distilbert_config.items()
])

config_table.to_csv(repo_config_table_path, index=False)
config_table.to_csv(drive_config_table_path, index=False)

print("config json saved:", repo_config_path)
print("config json backup saved:", drive_config_path)
print("config table saved:", repo_config_table_path)
print("config table backup saved:", drive_config_table_path)


# 8 final part 7 check
config_files_exist = (
    repo_config_path.exists()
    and drive_config_path.exists()
    and repo_config_table_path.exists()
    and drive_config_table_path.exists()
)

config_values_ok = (
    model_checkpoint == "distilbert-base-uncased"
    and num_labels == 2
    and max_length == 512
    and label2id["SAFE"] == 0
    and label2id["INJECTION"] == 1
    and positive_label == 1
)

print("final check")
print("config files exist:", config_files_exist)
print("config values ok:", config_values_ok)

if config_files_exist and config_values_ok:
    print("part 7 complete")
else:
    print("part 7 needs checking")

#Part 8 - Load DistilBERT Tokenizer

In [ ]:
# 1 import tokenizer class
from transformers import AutoTokenizer

print("tokenizer class imported")


# 2 load distilbert tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_checkpoint,
    use_fast=use_fast_tokenizer
)

print("tokenizer loaded")


# 3 check tokenizer information
print("tokenizer name:", model_checkpoint)
print("tokenizer class:", tokenizer.__class__.__name__)
print("fast tokenizer:", tokenizer.is_fast)
print("model max length:", tokenizer.model_max_length)
print("padding side:", tokenizer.padding_side)
print("truncation side:", tokenizer.truncation_side)


# 4 select one sample prompt
sample_text = train_df["text_for_model"].iloc[0]

print("sample text")
print(sample_text)


# 5 tokenize one sample prompt
sample_tokens = tokenizer(
    sample_text,
    max_length=max_length,
    padding=padding_strategy,
    truncation=truncation,
    return_tensors=None
)

print("sample tokenization complete")


# 6 inspect tokenizer output keys
print("tokenizer output keys:", sample_tokens.keys())


# 7 inspect input ids and attention mask
print("input_ids length:", len(sample_tokens["input_ids"]))
print("attention_mask length:", len(sample_tokens["attention_mask"]))

print("first 20 input_ids:")
print(sample_tokens["input_ids"][:20])

print("first 20 attention_mask values:")
print(sample_tokens["attention_mask"][:20])


# 8 convert first tokens back to readable tokens
first_20_tokens = tokenizer.convert_ids_to_tokens(sample_tokens["input_ids"][:20])

print("first 20 tokens:")
print(first_20_tokens)


# 9 check required tokenizer fields
has_input_ids = "input_ids" in sample_tokens
has_attention_mask = "attention_mask" in sample_tokens
length_ok = len(sample_tokens["input_ids"]) == max_length

print("tokenizer field check")
print("has input_ids:", has_input_ids)
print("has attention_mask:", has_attention_mask)
print("length equals max_length:", length_ok)


# 10 save tokenizer check summary
tokenizer_check_summary = pd.DataFrame([
    {"check": "tokenizer checkpoint", "value": model_checkpoint},
    {"check": "tokenizer class", "value": tokenizer.__class__.__name__},
    {"check": "fast tokenizer", "value": tokenizer.is_fast},
    {"check": "model max length", "value": tokenizer.model_max_length},
    {"check": "padding side", "value": tokenizer.padding_side},
    {"check": "truncation side", "value": tokenizer.truncation_side},
    {"check": "has input_ids", "value": has_input_ids},
    {"check": "has attention_mask", "value": has_attention_mask},
    {"check": "length equals max_length", "value": length_ok},
])

tokenizer_check_path = repo_tables_dir / "distilbert_tokenizer_check.csv"
drive_tokenizer_check_path = drive_tables_dir / "distilbert_tokenizer_check.csv"

tokenizer_check_summary.to_csv(tokenizer_check_path, index=False)
tokenizer_check_summary.to_csv(drive_tokenizer_check_path, index=False)

print("tokenizer check saved:", tokenizer_check_path)
print("tokenizer check backup saved:", drive_tokenizer_check_path)


# 11 final part 8 check
tokenizer_ok = has_input_ids and has_attention_mask and length_ok

print("final check")
print("tokenizer ok:", tokenizer_ok)

if tokenizer_ok:
    print("part 8 complete")
else:
    print("part 8 needs checking")

#Part 9 - Tokenize Train, Validation, and Test Data

In [ ]:
# 1 define tokenization function
def tokenize_texts(text_list):
    return tokenizer(
        text_list,
        max_length=max_length,
        padding=padding_strategy,
        truncation=truncation
    )

print("tokenization function ready")


# 2 tokenize train texts
train_tokenized = tokenize_texts(train_df["text_for_model"].astype(str).tolist())

print("train tokenized")


# 3 tokenize validation texts
validation_tokenized = tokenize_texts(validation_df["text_for_model"].astype(str).tolist())

print("validation tokenized")


# 4 tokenize test texts
test_tokenized = tokenize_texts(test_df["text_for_model"].astype(str).tolist())

print("test tokenized")


# 5 attach labels
train_tokenized["labels"] = train_df["label"].astype(int).tolist()
validation_tokenized["labels"] = validation_df["label"].astype(int).tolist()
test_tokenized["labels"] = test_df["label"].astype(int).tolist()

print("labels attached")


# 6 check tokenized dataset sizes
print("tokenized sizes")
print("train:", len(train_tokenized["input_ids"]))
print("validation:", len(validation_tokenized["input_ids"]))
print("test:", len(test_tokenized["input_ids"]))


# 7 check tokenized fields
print("tokenized fields")
print("train fields:", train_tokenized.keys())
print("validation fields:", validation_tokenized.keys())
print("test fields:", test_tokenized.keys())


# 8 check one tokenized example
example_index = 0

print("one train example check")
print("input_ids length:", len(train_tokenized["input_ids"][example_index]))
print("attention_mask length:", len(train_tokenized["attention_mask"][example_index]))
print("label:", train_tokenized["labels"][example_index])


# 9 check all sequence lengths
train_lengths_ok = all(len(ids) == max_length for ids in train_tokenized["input_ids"])
validation_lengths_ok = all(len(ids) == max_length for ids in validation_tokenized["input_ids"])
test_lengths_ok = all(len(ids) == max_length for ids in test_tokenized["input_ids"])

print("sequence length check")
print("train lengths ok:", train_lengths_ok)
print("validation lengths ok:", validation_lengths_ok)
print("test lengths ok:", test_lengths_ok)


# 10 check label counts after tokenization
print("label counts after tokenization")
print("train:", pd.Series(train_tokenized["labels"]).value_counts().sort_index().to_dict())
print("validation:", pd.Series(validation_tokenized["labels"]).value_counts().sort_index().to_dict())
print("test:", pd.Series(test_tokenized["labels"]).value_counts().sort_index().to_dict())


# 11 save tokenization summary
tokenization_summary = pd.DataFrame([
    {
        "split": "train",
        "rows": len(train_tokenized["input_ids"]),
        "has_input_ids": "input_ids" in train_tokenized,
        "has_attention_mask": "attention_mask" in train_tokenized,
        "has_labels": "labels" in train_tokenized,
        "all_lengths_equal_max_length": train_lengths_ok,
        "max_length": max_length
    },
    {
        "split": "validation",
        "rows": len(validation_tokenized["input_ids"]),
        "has_input_ids": "input_ids" in validation_tokenized,
        "has_attention_mask": "attention_mask" in validation_tokenized,
        "has_labels": "labels" in validation_tokenized,
        "all_lengths_equal_max_length": validation_lengths_ok,
        "max_length": max_length
    },
    {
        "split": "test",
        "rows": len(test_tokenized["input_ids"]),
        "has_input_ids": "input_ids" in test_tokenized,
        "has_attention_mask": "attention_mask" in test_tokenized,
        "has_labels": "labels" in test_tokenized,
        "all_lengths_equal_max_length": test_lengths_ok,
        "max_length": max_length
    }
])

tokenization_summary_path = repo_tables_dir / "distilbert_tokenization_summary.csv"
drive_tokenization_summary_path = drive_tables_dir / "distilbert_tokenization_summary.csv"

tokenization_summary.to_csv(tokenization_summary_path, index=False)
tokenization_summary.to_csv(drive_tokenization_summary_path, index=False)

print("tokenization summary saved:", tokenization_summary_path)
print("tokenization summary backup saved:", drive_tokenization_summary_path)


# 12 final part 9 check
tokenized_sizes_ok = (
    len(train_tokenized["input_ids"]) == len(train_df)
    and len(validation_tokenized["input_ids"]) == len(validation_df)
    and len(test_tokenized["input_ids"]) == len(test_df)
)

tokenized_fields_ok = (
    "input_ids" in train_tokenized
    and "attention_mask" in train_tokenized
    and "labels" in train_tokenized
    and "input_ids" in validation_tokenized
    and "attention_mask" in validation_tokenized
    and "labels" in validation_tokenized
    and "input_ids" in test_tokenized
    and "attention_mask" in test_tokenized
    and "labels" in test_tokenized
)

all_lengths_ok = train_lengths_ok and validation_lengths_ok and test_lengths_ok

print("final check")
print("tokenized sizes ok:", tokenized_sizes_ok)
print("tokenized fields ok:", tokenized_fields_ok)
print("all sequence lengths ok:", all_lengths_ok)

if tokenized_sizes_ok and tokenized_fields_ok and all_lengths_ok:
    print("part 9 complete")
else:
    print("part 9 needs checking")

#Part 10 - Create Hugging Face Dataset Objects

In [ ]:
# 1 disable torchvision check
import datasets.config as datasets_config

datasets_config.TORCHVISION_AVAILABLE = False


# 2 remove token_type_ids because distilbert does not need it
for data_name, tokenized_data in {
    "train": train_tokenized,
    "validation": validation_tokenized,
    "test": test_tokenized
}.items():
    if "token_type_ids" in tokenized_data:
        tokenized_data.pop("token_type_ids")
        print(data_name, "token_type_ids removed")
    else:
        print(data_name, "token_type_ids not found")


# 3 import hugging face dataset
from datasets import Dataset

print("dataset class imported")


# 4 create hugging face datasets
train_dataset = Dataset.from_dict(train_tokenized)
validation_dataset = Dataset.from_dict(validation_tokenized)
test_dataset = Dataset.from_dict(test_tokenized)

print("hugging face datasets created")


# 5 check dataset sizes
print("dataset sizes")
print("train:", len(train_dataset))
print("validation:", len(validation_dataset))
print("test:", len(test_dataset))


# 6 check dataset columns
print("dataset columns")
print("train:", train_dataset.column_names)
print("validation:", validation_dataset.column_names)
print("test:", test_dataset.column_names)


# 7 set format for pytorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
validation_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("torch format set")


# 8 inspect one training example
sample_item = train_dataset[0]

print("sample item keys:", sample_item.keys())
print("input_ids shape:", sample_item["input_ids"].shape)
print("attention_mask shape:", sample_item["attention_mask"].shape)
print("label:", sample_item["labels"])


# 9 check required fields
required_dataset_fields = ["input_ids", "attention_mask", "labels"]

def check_dataset_fields(dataset, dataset_name):
    fields_ok = all(field in dataset.column_names for field in required_dataset_fields)
    print(dataset_name, "fields ok:", fields_ok)
    return fields_ok

train_fields_ok = check_dataset_fields(train_dataset, "train")
validation_fields_ok = check_dataset_fields(validation_dataset, "validation")
test_fields_ok = check_dataset_fields(test_dataset, "test")


# 10 check dataset lengths
train_length_ok = len(train_dataset) == len(train_df)
validation_length_ok = len(validation_dataset) == len(validation_df)
test_length_ok = len(test_dataset) == len(test_df)

print("length check")
print("train length ok:", train_length_ok)
print("validation length ok:", validation_length_ok)
print("test length ok:", test_length_ok)


# 11 save dataset summary
hf_dataset_summary = pd.DataFrame([
    {
        "split": "train",
        "rows": len(train_dataset),
        "columns": ", ".join(train_dataset.column_names),
        "torch_format_ready": train_fields_ok,
    },
    {
        "split": "validation",
        "rows": len(validation_dataset),
        "columns": ", ".join(validation_dataset.column_names),
        "torch_format_ready": validation_fields_ok,
    },
    {
        "split": "test",
        "rows": len(test_dataset),
        "columns": ", ".join(test_dataset.column_names),
        "torch_format_ready": test_fields_ok,
    },
])

hf_dataset_summary_path = repo_tables_dir / "distilbert_hf_dataset_summary.csv"
drive_hf_dataset_summary_path = drive_tables_dir / "distilbert_hf_dataset_summary.csv"

hf_dataset_summary.to_csv(hf_dataset_summary_path, index=False)
hf_dataset_summary.to_csv(drive_hf_dataset_summary_path, index=False)

print("hf dataset summary saved:", hf_dataset_summary_path)
print("hf dataset summary backup saved:", drive_hf_dataset_summary_path)


# 12 final part 10 check
all_fields_ok = train_fields_ok and validation_fields_ok and test_fields_ok
all_lengths_ok = train_length_ok and validation_length_ok and test_length_ok

print("final check")
print("all fields ok:", all_fields_ok)
print("all lengths ok:", all_lengths_ok)

if all_fields_ok and all_lengths_ok:
    print("part 10 complete")
else:
    print("part 10 needs checking")

#Part 11 - Load DistilBERT for Binary Classification

In [ ]:
# 1 import model class
from transformers import AutoModelForSequenceClassification

print("model class imported")


# 2 load distilbert model for classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    problem_type=problem_type
)

print("model loaded")


# 3 move model to gpu
model.to(device)

print("model moved to device:", device)


# 4 check model information
print("model checkpoint:", model_checkpoint)
print("model class:", model.__class__.__name__)
print("number of labels:", model.config.num_labels)
print("id2label:", model.config.id2label)
print("label2id:", model.config.label2id)
print("problem type:", model.config.problem_type)


# 5 count model parameters
total_params = sum(param.numel() for param in model.parameters())
trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)

print("total parameters:", total_params)
print("trainable parameters:", trainable_params)


# 6 check classifier layer
print("classifier layer")
print(model.classifier)


# 7 run one small forward pass
sample_batch = {
    "input_ids": train_dataset[0]["input_ids"].unsqueeze(0).to(device),
    "attention_mask": train_dataset[0]["attention_mask"].unsqueeze(0).to(device)
}

with torch.no_grad():
    sample_output = model(**sample_batch)

print("sample forward pass complete")
print("logits shape:", sample_output.logits.shape)
print("logits:", sample_output.logits.detach().cpu().numpy())


# 8 check output shape
expected_output_shape = (1, num_labels)
actual_output_shape = tuple(sample_output.logits.shape)

output_shape_ok = actual_output_shape == expected_output_shape

print("output shape check")
print("expected:", expected_output_shape)
print("actual:", actual_output_shape)
print("output shape ok:", output_shape_ok)


# 9 save model loading summary
model_loading_summary = pd.DataFrame([
    {"check": "model checkpoint", "value": model_checkpoint},
    {"check": "model class", "value": model.__class__.__name__},
    {"check": "number of labels", "value": model.config.num_labels},
    {"check": "problem type", "value": model.config.problem_type},
    {"check": "device", "value": str(device)},
    {"check": "total parameters", "value": total_params},
    {"check": "trainable parameters", "value": trainable_params},
    {"check": "output shape", "value": str(actual_output_shape)},
    {"check": "output shape ok", "value": output_shape_ok},
])

model_loading_summary_path = repo_tables_dir / "distilbert_model_loading_summary.csv"
drive_model_loading_summary_path = drive_tables_dir / "distilbert_model_loading_summary.csv"

model_loading_summary.to_csv(model_loading_summary_path, index=False)
model_loading_summary.to_csv(drive_model_loading_summary_path, index=False)

print("model loading summary saved:", model_loading_summary_path)
print("model loading summary backup saved:", drive_model_loading_summary_path)


# 10 final part 11 check
model_loaded_ok = (
    model.config.num_labels == num_labels
    and model.config.id2label == id2label
    and output_shape_ok
)

print("final check")
print("model loaded ok:", model_loaded_ok)

if model_loaded_ok:
    print("part 11 complete")
else:
    print("part 11 needs checking")

#Part 12 - Define Evaluation Metrics


In [ ]:
# 1 import metric tools
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

print("metric tools imported")


# 2 define softmax function
def softmax(logits):
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probabilities = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
    return probabilities

print("softmax function ready")


# 3 define metric function for trainer
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probabilities = softmax(logits)
    predictions = np.argmax(probabilities, axis=1)

    probability_injection = probabilities[:, positive_label]

    accuracy = accuracy_score(labels, predictions)

    precision = precision_score(
        labels,
        predictions,
        pos_label=positive_label,
        zero_division=0
    )

    recall = recall_score(
        labels,
        predictions,
        pos_label=positive_label,
        zero_division=0
    )

    f1 = f1_score(
        labels,
        predictions,
        pos_label=positive_label,
        zero_division=0
    )

    unique_labels = np.unique(labels)

    if len(unique_labels) == 2:
        auc_roc = roc_auc_score(labels, probability_injection)
    else:
        auc_roc = np.nan

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc_roc": auc_roc
    }

print("compute_metrics function ready")


# 4 test metric function with small fake prediction example
test_logits = np.array([
    [2.0, 0.5],
    [0.3, 1.8],
    [1.5, 0.2],
    [0.4, 2.1]
])

test_labels = np.array([0, 1, 0, 1])

test_metrics = compute_metrics((test_logits, test_labels))

print("test metrics")
print(test_metrics)


# 5 check metric keys
required_metric_keys = ["accuracy", "precision", "recall", "f1", "auc_roc"]

metric_keys_ok = all(key in test_metrics for key in required_metric_keys)

print("metric keys ok:", metric_keys_ok)


# 6 save metric definition summary
metric_summary = pd.DataFrame([
    {
        "metric": "accuracy",
        "meaning": "overall proportion of correct predictions",
        "used_for_selection": False
    },
    {
        "metric": "precision",
        "meaning": "of prompts predicted as INJECTION, proportion that are actually INJECTION",
        "used_for_selection": False
    },
    {
        "metric": "recall",
        "meaning": "of real INJECTION prompts, proportion detected by the model",
        "used_for_selection": False
    },
    {
        "metric": "f1",
        "meaning": "balance between precision and recall",
        "used_for_selection": True
    },
    {
        "metric": "auc_roc",
        "meaning": "ability to separate SAFE and INJECTION using prediction probabilities",
        "used_for_selection": False
    }
])

metric_summary_path = repo_tables_dir / "distilbert_metric_definition_summary.csv"
drive_metric_summary_path = drive_tables_dir / "distilbert_metric_definition_summary.csv"

metric_summary.to_csv(metric_summary_path, index=False)
metric_summary.to_csv(drive_metric_summary_path, index=False)

print("metric summary saved:", metric_summary_path)
print("metric summary backup saved:", drive_metric_summary_path)


# 7 save positive class information
positive_class_summary = pd.DataFrame([
    {
        "setting": "positive_label",
        "value": positive_label
    },
    {
        "setting": "positive_label_name",
        "value": positive_label_name
    },
    {
        "setting": "reason",
        "value": "INJECTION is the positive class because the security goal is to detect malicious prompt-injection behaviour."
    },
    {
        "setting": "false_negative_meaning",
        "value": "A real INJECTION prompt predicted as SAFE."
    },
    {
        "setting": "false_negative_risk",
        "value": "False negatives are risky because malicious prompts may pass through the detector."
    }
])

positive_class_path = repo_tables_dir / "distilbert_positive_class_summary.csv"
drive_positive_class_path = drive_tables_dir / "distilbert_positive_class_summary.csv"

positive_class_summary.to_csv(positive_class_path, index=False)
positive_class_summary.to_csv(drive_positive_class_path, index=False)

print("positive class summary saved:", positive_class_path)
print("positive class summary backup saved:", drive_positive_class_path)


# 8 final part 12 check
metrics_function_ok = metric_keys_ok and callable(compute_metrics)

summary_files_exist = (
    metric_summary_path.exists()
    and drive_metric_summary_path.exists()
    and positive_class_path.exists()
    and drive_positive_class_path.exists()
)

print("final check")
print("metrics function ok:", metrics_function_ok)
print("summary files exist:", summary_files_exist)

if metrics_function_ok and summary_files_exist:
    print("part 12 complete")
else:
    print("part 12 needs checking")

#Part 13 - Define Training Arguments


In [ ]:
# 1 import training arguments
from transformers import TrainingArguments
import inspect
import math

print("training arguments class imported")


# 2 define basic training settings
num_train_epochs = 4
learning_rate = 2e-5
weight_decay = 0.01

per_device_train_batch_size = 8
per_device_eval_batch_size = 16

gradient_accumulation_steps = 1
warmup_ratio = 0.1

logging_steps = 10
save_total_limit = 2

use_fp16 = torch.cuda.is_available()

print("basic training settings ready")


# 3 estimate training steps
train_rows = len(train_dataset)
steps_per_epoch = math.ceil(train_rows / per_device_train_batch_size)
estimated_total_steps = steps_per_epoch * num_train_epochs

print("training step estimate")
print("train rows:", train_rows)
print("batch size:", per_device_train_batch_size)
print("steps per epoch:", steps_per_epoch)
print("epochs:", num_train_epochs)
print("estimated total steps:", estimated_total_steps)


# 4 define training output folders
training_output_dir = Path("/content/distilbert_training_output")
training_logging_dir = repo_logs_dir / "trainer_logs"

training_output_dir.mkdir(parents=True, exist_ok=True)
training_logging_dir.mkdir(parents=True, exist_ok=True)

print("training output dir:", training_output_dir)
print("training logging dir:", training_logging_dir)


# 5 handle transformers version difference
training_args_signature = inspect.signature(TrainingArguments.__init__)
training_args_parameters = training_args_signature.parameters

if "eval_strategy" in training_args_parameters:
    eval_strategy_key = "eval_strategy"
else:
    eval_strategy_key = "evaluation_strategy"

print("evaluation strategy argument name:", eval_strategy_key)


# 6 build training arguments dictionary
training_args_dict = {
    "output_dir": str(training_output_dir),
    "num_train_epochs": num_train_epochs,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "per_device_train_batch_size": per_device_train_batch_size,
    "per_device_eval_batch_size": per_device_eval_batch_size,
    "gradient_accumulation_steps": gradient_accumulation_steps,
    "warmup_ratio": warmup_ratio,
    eval_strategy_key: "epoch",
    "save_strategy": "epoch",
    "logging_strategy": "steps",
    "logging_steps": logging_steps,
    "load_best_model_at_end": True,
    "metric_for_best_model": main_selection_metric,
    "greater_is_better": greater_is_better,
    "save_total_limit": save_total_limit,
    "seed": random_seed,
    "data_seed": random_seed,
    "fp16": use_fp16,
    "report_to": "none",
    "logging_dir": str(training_logging_dir)
}

print("training arguments dictionary ready")


# 7 create training arguments
training_args = TrainingArguments(**training_args_dict)

print("training arguments created")


# 8 print key training settings
print("training settings")
print("output dir:", training_args.output_dir)
print("epochs:", training_args.num_train_epochs)
print("learning rate:", training_args.learning_rate)
print("weight decay:", training_args.weight_decay)
print("train batch size:", training_args.per_device_train_batch_size)
print("eval batch size:", training_args.per_device_eval_batch_size)
print("fp16:", training_args.fp16)
print("best model metric:", training_args.metric_for_best_model)
print("greater is better:", training_args.greater_is_better)
print("load best model at end:", training_args.load_best_model_at_end)
print("save total limit:", training_args.save_total_limit)


# 9 save training arguments summary
training_args_summary = pd.DataFrame([
    {"setting": "model_name", "value": model_name},
    {"setting": "model_checkpoint", "value": model_checkpoint},
    {"setting": "output_dir", "value": str(training_output_dir)},
    {"setting": "num_train_epochs", "value": num_train_epochs},
    {"setting": "learning_rate", "value": learning_rate},
    {"setting": "weight_decay", "value": weight_decay},
    {"setting": "per_device_train_batch_size", "value": per_device_train_batch_size},
    {"setting": "per_device_eval_batch_size", "value": per_device_eval_batch_size},
    {"setting": "gradient_accumulation_steps", "value": gradient_accumulation_steps},
    {"setting": "warmup_ratio", "value": warmup_ratio},
    {"setting": "evaluation_strategy", "value": "epoch"},
    {"setting": "save_strategy", "value": "epoch"},
    {"setting": "logging_strategy", "value": "steps"},
    {"setting": "logging_steps", "value": logging_steps},
    {"setting": "load_best_model_at_end", "value": True},
    {"setting": "metric_for_best_model", "value": main_selection_metric},
    {"setting": "greater_is_better", "value": greater_is_better},
    {"setting": "save_total_limit", "value": save_total_limit},
    {"setting": "seed", "value": random_seed},
    {"setting": "fp16", "value": use_fp16},
    {"setting": "train_rows", "value": train_rows},
    {"setting": "steps_per_epoch", "value": steps_per_epoch},
    {"setting": "estimated_total_steps", "value": estimated_total_steps},
])

training_args_summary_path = repo_tables_dir / "distilbert_training_arguments_summary.csv"
drive_training_args_summary_path = drive_tables_dir / "distilbert_training_arguments_summary.csv"

training_args_summary.to_csv(training_args_summary_path, index=False)
training_args_summary.to_csv(drive_training_args_summary_path, index=False)

print("training arguments summary saved:", training_args_summary_path)
print("training arguments summary backup saved:", drive_training_args_summary_path)


# 10 final part 13 check
training_args_ok = (
    training_args.output_dir == str(training_output_dir)
    and training_args.num_train_epochs == num_train_epochs
    and training_args.per_device_train_batch_size == per_device_train_batch_size
    and training_args.metric_for_best_model == main_selection_metric
    and training_args.load_best_model_at_end == True
)

summary_files_exist = (
    training_args_summary_path.exists()
    and drive_training_args_summary_path.exists()
)

print("final check")
print("training args ok:", training_args_ok)
print("summary files exist:", summary_files_exist)

if training_args_ok and summary_files_exist:
    print("part 13 complete")
else:
    print("part 13 needs checking")

#Part 14 - Configuration of Trainer

In [ ]:
# 1 import trainer tools
from transformers import Trainer
from transformers import EarlyStoppingCallback
import inspect

print("trainer tools imported")


# 2 define early stopping
early_stopping_patience = 2

early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=early_stopping_patience
)

print("early stopping ready")
print("early stopping patience:", early_stopping_patience)


# 3 prepare trainer arguments
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": validation_dataset,
    "compute_metrics": compute_metrics,
    "callbacks": [early_stopping_callback],
}

print("basic trainer arguments ready")


# 4 handle transformers tokenizer argument
trainer_signature = inspect.signature(Trainer.__init__)
trainer_parameters = trainer_signature.parameters

if "processing_class" in trainer_parameters:
    trainer_kwargs["processing_class"] = tokenizer
    print("using processing_class for tokenizer")
elif "tokenizer" in trainer_parameters:
    trainer_kwargs["tokenizer"] = tokenizer
    print("using tokenizer argument")
else:
    print("trainer does not need tokenizer argument")


# 5 create trainer
trainer = Trainer(**trainer_kwargs)

print("trainer created")


# 6 check trainer datasets
print("trainer dataset check")
print("train dataset rows:", len(trainer.train_dataset))
print("validation dataset rows:", len(trainer.eval_dataset))


# 7 check trainer model and args
print("trainer model check")
print("model class:", trainer.model.__class__.__name__)
print("model device:", next(trainer.model.parameters()).device)

print("trainer args check")
print("output dir:", trainer.args.output_dir)
print("epochs:", trainer.args.num_train_epochs)
print("train batch size:", trainer.args.per_device_train_batch_size)
print("eval batch size:", trainer.args.per_device_eval_batch_size)
print("best metric:", trainer.args.metric_for_best_model)
print("load best model at end:", trainer.args.load_best_model_at_end)


# 8 check callbacks
callback_names = [callback.__class__.__name__ for callback in trainer.callback_handler.callbacks]

print("trainer callbacks")
print(callback_names)

early_stopping_added = "EarlyStoppingCallback" in callback_names

print("early stopping added:", early_stopping_added)


# 9 run a very small trainer sanity check
# this does not train the model
sample_train_item = trainer.train_dataset[0]
sample_eval_item = trainer.eval_dataset[0]

sample_check_ok = (
    "input_ids" in sample_train_item
    and "attention_mask" in sample_train_item
    and "labels" in sample_train_item
    and "input_ids" in sample_eval_item
    and "attention_mask" in sample_eval_item
    and "labels" in sample_eval_item
)

print("sample dataset check ok:", sample_check_ok)


# 10 save trainer configuration summary
trainer_summary = pd.DataFrame([
    {"setting": "trainer_created", "value": True},
    {"setting": "model_class", "value": trainer.model.__class__.__name__},
    {"setting": "model_device", "value": str(next(trainer.model.parameters()).device)},
    {"setting": "train_dataset_rows", "value": len(trainer.train_dataset)},
    {"setting": "validation_dataset_rows", "value": len(trainer.eval_dataset)},
    {"setting": "output_dir", "value": trainer.args.output_dir},
    {"setting": "num_train_epochs", "value": trainer.args.num_train_epochs},
    {"setting": "train_batch_size", "value": trainer.args.per_device_train_batch_size},
    {"setting": "eval_batch_size", "value": trainer.args.per_device_eval_batch_size},
    {"setting": "learning_rate", "value": trainer.args.learning_rate},
    {"setting": "weight_decay", "value": trainer.args.weight_decay},
    {"setting": "metric_for_best_model", "value": trainer.args.metric_for_best_model},
    {"setting": "load_best_model_at_end", "value": trainer.args.load_best_model_at_end},
    {"setting": "early_stopping_added", "value": early_stopping_added},
    {"setting": "early_stopping_patience", "value": early_stopping_patience},
    {"setting": "sample_check_ok", "value": sample_check_ok},
])

trainer_summary_path = repo_tables_dir / "distilbert_trainer_configuration_summary.csv"
drive_trainer_summary_path = drive_tables_dir / "distilbert_trainer_configuration_summary.csv"

trainer_summary.to_csv(trainer_summary_path, index=False)
trainer_summary.to_csv(drive_trainer_summary_path, index=False)

print("trainer summary saved:", trainer_summary_path)
print("trainer summary backup saved:", drive_trainer_summary_path)


# 11 final part 14 check
trainer_ready = (
    trainer is not None
    and len(trainer.train_dataset) == len(train_dataset)
    and len(trainer.eval_dataset) == len(validation_dataset)
    and trainer.args.metric_for_best_model == main_selection_metric
    and early_stopping_added
    and sample_check_ok
)

summary_files_exist = (
    trainer_summary_path.exists()
    and drive_trainer_summary_path.exists()
)

print("final check")
print("trainer ready:", trainer_ready)
print("summary files exist:", summary_files_exist)

if trainer_ready and summary_files_exist:
    print("part 14 complete")
else:
    print("part 14 needs checking")

#Part 15 — Reset running memory

In [ ]:
# 1 clean reset before official distilbert training run
import shutil
import gc

print("cleaning old training output")

if training_output_dir.exists():
    shutil.rmtree(training_output_dir)
    print("old training output removed:", training_output_dir)
else:
    print("no old training output found")

training_output_dir.mkdir(parents=True, exist_ok=True)
print("fresh training output folder created:", training_output_dir)

if "model" in globals():
    del model
    print("old model removed from memory")

if "trainer" in globals():
    del trainer
    print("old trainer removed from memory")

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("gpu cache cleared")

print("clean reset complete")

#Part 15 - Fine-Tune DistilBERT


In [ ]:
# 1 check trainer before training
print("training check")
print("model device:", next(trainer.model.parameters()).device)
print("train rows:", len(trainer.train_dataset))
print("validation rows:", len(trainer.eval_dataset))
print("epochs:", trainer.args.num_train_epochs)
print("batch size:", trainer.args.per_device_train_batch_size)
print("best metric:", trainer.args.metric_for_best_model)


# 2 clear gpu cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("gpu cache cleared")
else:
    print("gpu not available, training will use cpu")


# 3 record start time
training_start_time = time.time()

print("training started")


# 4 train model
train_result = trainer.train()

print("training finished")


# 5 record end time
training_end_time = time.time()
training_time_seconds = training_end_time - training_start_time
training_time_minutes = training_time_seconds / 60

print("training time seconds:", round(training_time_seconds, 2))
print("training time minutes:", round(training_time_minutes, 2))


# 6 collect training metrics
train_metrics = train_result.metrics

train_metrics["training_time_seconds"] = round(training_time_seconds, 2)
train_metrics["training_time_minutes"] = round(training_time_minutes, 2)
train_metrics["train_rows"] = len(train_dataset)
train_metrics["validation_rows"] = len(validation_dataset)
train_metrics["epochs"] = num_train_epochs
train_metrics["batch_size"] = per_device_train_batch_size
train_metrics["learning_rate"] = learning_rate
train_metrics["best_metric"] = main_selection_metric

print("training metrics")
print(train_metrics)


# 7 save training metrics
training_metrics_df = pd.DataFrame([
    {"metric": key, "value": value}
    for key, value in train_metrics.items()
])

training_metrics_path = repo_tables_dir / "distilbert_training_metrics.csv"
drive_training_metrics_path = drive_tables_dir / "distilbert_training_metrics.csv"

training_metrics_df.to_csv(training_metrics_path, index=False)
training_metrics_df.to_csv(drive_training_metrics_path, index=False)

print("training metrics saved:", training_metrics_path)
print("training metrics backup saved:", drive_training_metrics_path)


# 8 save trainer state
trainer_state_path = repo_logs_dir / "distilbert_trainer_state.json"
drive_trainer_state_path = drive_logs_dir / "distilbert_trainer_state.json"

trainer.state.save_to_json(str(trainer_state_path))
trainer.state.save_to_json(str(drive_trainer_state_path))

print("trainer state saved:", trainer_state_path)
print("trainer state backup saved:", drive_trainer_state_path)


# 9 get best checkpoint information
best_checkpoint = trainer.state.best_model_checkpoint
best_metric_value = trainer.state.best_metric

print("best checkpoint:", best_checkpoint)
print("best metric value:", best_metric_value)


# 10 save best checkpoint summary
best_checkpoint_summary = pd.DataFrame([
    {"item": "best_checkpoint", "value": str(best_checkpoint)},
    {"item": "best_metric_name", "value": main_selection_metric},
    {"item": "best_metric_value", "value": best_metric_value},
    {"item": "training_time_seconds", "value": round(training_time_seconds, 2)},
    {"item": "training_time_minutes", "value": round(training_time_minutes, 2)},
])

best_checkpoint_path = repo_tables_dir / "distilbert_best_checkpoint_summary.csv"
drive_best_checkpoint_path = drive_tables_dir / "distilbert_best_checkpoint_summary.csv"

best_checkpoint_summary.to_csv(best_checkpoint_path, index=False)
best_checkpoint_summary.to_csv(drive_best_checkpoint_path, index=False)

print("best checkpoint summary saved:", best_checkpoint_path)
print("best checkpoint summary backup saved:", drive_best_checkpoint_path)


# 11 final part 15 check
training_metrics_saved = training_metrics_path.exists() and drive_training_metrics_path.exists()
trainer_state_saved = trainer_state_path.exists() and drive_trainer_state_path.exists()
best_checkpoint_saved = best_checkpoint_path.exists() and drive_best_checkpoint_path.exists()
best_checkpoint_exists = best_checkpoint is not None

print("final check")
print("training metrics saved:", training_metrics_saved)
print("trainer state saved:", trainer_state_saved)
print("best checkpoint summary saved:", best_checkpoint_saved)
print("best checkpoint exists:", best_checkpoint_exists)

if training_metrics_saved and trainer_state_saved and best_checkpoint_saved and best_checkpoint_exists:
    print("part 15 complete")
else:
    print("part 15 needs checking")

#Part 16 - Save Training Log and Best Model


In [ ]:
# 1 define model save folders
best_model_dir = drive_model_dir / "best_model"
tokenizer_save_dir = drive_model_dir / "tokenizer"

best_model_dir.mkdir(parents=True, exist_ok=True)
tokenizer_save_dir.mkdir(parents=True, exist_ok=True)

print("model save folders ready")
print("best model dir:", best_model_dir)
print("tokenizer dir:", tokenizer_save_dir)


# 2 confirm best checkpoint before saving
best_checkpoint = trainer.state.best_model_checkpoint
best_metric_value = trainer.state.best_metric

print("best checkpoint:", best_checkpoint)
print("best metric value:", best_metric_value)


# 3 save best model to google drive
# because load_best_model_at_end=True, trainer.model is already the best validation model
trainer.save_model(str(best_model_dir))

print("best model saved to drive")


# 4 save tokenizer
tokenizer.save_pretrained(str(best_model_dir))
tokenizer.save_pretrained(str(tokenizer_save_dir))

print("tokenizer saved")


# 5 save model config
model.config.save_pretrained(str(best_model_dir))

print("model config saved")


# 6 save trainer log history
trainer_log_history = pd.DataFrame(trainer.state.log_history)

trainer_log_history_path = repo_logs_dir / "distilbert_trainer_log_history.csv"
drive_trainer_log_history_path = drive_logs_dir / "distilbert_trainer_log_history.csv"

trainer_log_history.to_csv(trainer_log_history_path, index=False)
trainer_log_history.to_csv(drive_trainer_log_history_path, index=False)

print("trainer log history saved:", trainer_log_history_path)
print("trainer log history backup saved:", drive_trainer_log_history_path)


# 7 save final trainer state
trainer_state_path = repo_logs_dir / "distilbert_trainer_state_final.json"
drive_trainer_state_path = drive_logs_dir / "distilbert_trainer_state_final.json"

trainer.state.save_to_json(str(trainer_state_path))
trainer.state.save_to_json(str(drive_trainer_state_path))

print("trainer state saved:", trainer_state_path)
print("trainer state backup saved:", drive_trainer_state_path)


# 8 list saved files
saved_files = sorted([file.name for file in best_model_dir.iterdir() if file.is_file()])
tokenizer_files = sorted([file.name for file in tokenizer_save_dir.iterdir() if file.is_file()])

print("saved model files")
print(saved_files)

print("saved tokenizer files")
print(tokenizer_files)


# 9 check important saved files
config_exists = (best_model_dir / "config.json").exists()

model_safetensors_exists = (best_model_dir / "model.safetensors").exists()
pytorch_model_exists = (best_model_dir / "pytorch_model.bin").exists()
model_weights_exist = model_safetensors_exists or pytorch_model_exists

tokenizer_config_exists = (best_model_dir / "tokenizer_config.json").exists()
tokenizer_json_exists = (best_model_dir / "tokenizer.json").exists()
vocab_exists = (best_model_dir / "vocab.txt").exists()

# fast tokenizers may use tokenizer.json instead of vocab.txt
tokenizer_files_ok = tokenizer_config_exists and (tokenizer_json_exists or vocab_exists)

print("file check")
print("config exists:", config_exists)
print("model.safetensors exists:", model_safetensors_exists)
print("pytorch_model.bin exists:", pytorch_model_exists)
print("model weights exist:", model_weights_exist)
print("tokenizer config exists:", tokenizer_config_exists)
print("tokenizer.json exists:", tokenizer_json_exists)
print("vocab.txt exists:", vocab_exists)
print("tokenizer files ok:", tokenizer_files_ok)


# 10 save model save summary
model_save_summary = pd.DataFrame([
    {"item": "model_name", "value": model_name},
    {"item": "model_checkpoint", "value": model_checkpoint},
    {"item": "best_checkpoint", "value": str(best_checkpoint)},
    {"item": "best_metric_name", "value": main_selection_metric},
    {"item": "best_metric_value", "value": best_metric_value},
    {"item": "best_model_dir", "value": str(best_model_dir)},
    {"item": "tokenizer_save_dir", "value": str(tokenizer_save_dir)},
    {"item": "config_exists", "value": config_exists},
    {"item": "model_safetensors_exists", "value": model_safetensors_exists},
    {"item": "pytorch_model_exists", "value": pytorch_model_exists},
    {"item": "model_weights_exist", "value": model_weights_exist},
    {"item": "tokenizer_config_exists", "value": tokenizer_config_exists},
    {"item": "tokenizer_json_exists", "value": tokenizer_json_exists},
    {"item": "vocab_exists", "value": vocab_exists},
    {"item": "tokenizer_files_ok", "value": tokenizer_files_ok},
    {"item": "saved_model_files", "value": ", ".join(saved_files)},
    {"item": "saved_tokenizer_files", "value": ", ".join(tokenizer_files)},
])

model_save_summary_path = repo_tables_dir / "distilbert_model_save_summary.csv"
drive_model_save_summary_path = drive_tables_dir / "distilbert_model_save_summary.csv"

model_save_summary.to_csv(model_save_summary_path, index=False)
model_save_summary.to_csv(drive_model_save_summary_path, index=False)

print("model save summary saved:", model_save_summary_path)
print("model save summary backup saved:", drive_model_save_summary_path)


# 11 final part 16 check
logs_saved = (
    trainer_log_history_path.exists()
    and drive_trainer_log_history_path.exists()
    and trainer_state_path.exists()
    and drive_trainer_state_path.exists()
)

summary_saved = (
    model_save_summary_path.exists()
    and drive_model_save_summary_path.exists()
)

model_saved_ok = config_exists and model_weights_exist and tokenizer_files_ok

print("final check")
print("model saved ok:", model_saved_ok)
print("logs saved:", logs_saved)
print("summary saved:", summary_saved)

if model_saved_ok and logs_saved and summary_saved:
    print("part 16 complete")
else:
    print("part 16 needs checking")

#Part 17 - Evaluate DistilBERT on Validation Set

In [ ]:
# 1 run prediction on validation set
print("running validation prediction")

validation_prediction_output = trainer.predict(
    validation_dataset,
    metric_key_prefix="validation"
)

print("validation prediction complete")


# 2 collect validation outputs
validation_logits = validation_prediction_output.predictions
validation_true_labels = validation_prediction_output.label_ids
validation_metrics = validation_prediction_output.metrics

print("validation logits shape:", validation_logits.shape)
print("validation labels shape:", validation_true_labels.shape)
print("validation metrics")
print(validation_metrics)


# 3 convert logits to probabilities and predictions
validation_probabilities = softmax(validation_logits)
validation_predicted_labels = np.argmax(validation_probabilities, axis=1)

validation_probability_safe = validation_probabilities[:, 0]
validation_probability_injection = validation_probabilities[:, 1]

print("validation probabilities created")
print("validation predictions created")


# 4 create validation metrics table
validation_metrics_clean = {
    "validation_loss": validation_metrics.get("validation_loss"),
    "validation_accuracy": validation_metrics.get("validation_accuracy"),
    "validation_precision": validation_metrics.get("validation_precision"),
    "validation_recall": validation_metrics.get("validation_recall"),
    "validation_f1": validation_metrics.get("validation_f1"),
    "validation_auc_roc": validation_metrics.get("validation_auc_roc"),
    "validation_runtime": validation_metrics.get("validation_runtime"),
    "validation_samples_per_second": validation_metrics.get("validation_samples_per_second"),
    "validation_steps_per_second": validation_metrics.get("validation_steps_per_second"),
}

validation_metrics_df = pd.DataFrame([
    {"metric": key, "value": value}
    for key, value in validation_metrics_clean.items()
])

print("validation metrics table created")
display(validation_metrics_df)


# 5 create validation prediction table
validation_predictions_df = validation_df.copy()

validation_predictions_df["true_label"] = validation_true_labels
validation_predictions_df["true_label_name"] = validation_predictions_df["true_label"].map(id2label)

validation_predictions_df["predicted_label"] = validation_predicted_labels
validation_predictions_df["predicted_label_name"] = validation_predictions_df["predicted_label"].map(id2label)

validation_predictions_df["probability_SAFE"] = validation_probability_safe
validation_predictions_df["probability_INJECTION"] = validation_probability_injection

validation_predictions_df["correct_or_incorrect"] = np.where(
    validation_predictions_df["true_label"] == validation_predictions_df["predicted_label"],
    "correct",
    "incorrect"
)

print("validation prediction table created")
display(validation_predictions_df.head())


# 6 check validation prediction counts
validation_correct_count = (validation_predictions_df["correct_or_incorrect"] == "correct").sum()
validation_incorrect_count = (validation_predictions_df["correct_or_incorrect"] == "incorrect").sum()

print("validation prediction count")
print("correct:", validation_correct_count)
print("incorrect:", validation_incorrect_count)


# 7 save validation metrics and predictions
validation_metrics_path = repo_tables_dir / "distilbert_validation_metrics.csv"
drive_validation_metrics_path = drive_tables_dir / "distilbert_validation_metrics.csv"

validation_predictions_path = repo_predictions_dir / "distilbert_validation_predictions.csv"
drive_validation_predictions_path = drive_predictions_dir / "distilbert_validation_predictions.csv"

validation_metrics_df.to_csv(validation_metrics_path, index=False)
validation_metrics_df.to_csv(drive_validation_metrics_path, index=False)

validation_predictions_df.to_csv(validation_predictions_path, index=False)
validation_predictions_df.to_csv(drive_validation_predictions_path, index=False)

print("validation metrics saved:", validation_metrics_path)
print("validation metrics backup saved:", drive_validation_metrics_path)

print("validation predictions saved:", validation_predictions_path)
print("validation predictions backup saved:", drive_validation_predictions_path)


# 8 save validation evaluation summary
validation_evaluation_summary = pd.DataFrame([
    {"item": "model_name", "value": model_name},
    {"item": "model_checkpoint", "value": model_checkpoint},
    {"item": "evaluation_split", "value": "validation"},
    {"item": "validation_rows", "value": len(validation_df)},
    {"item": "correct_predictions", "value": validation_correct_count},
    {"item": "incorrect_predictions", "value": validation_incorrect_count},
    {"item": "accuracy", "value": validation_metrics_clean["validation_accuracy"]},
    {"item": "precision", "value": validation_metrics_clean["validation_precision"]},
    {"item": "recall", "value": validation_metrics_clean["validation_recall"]},
    {"item": "f1", "value": validation_metrics_clean["validation_f1"]},
    {"item": "auc_roc", "value": validation_metrics_clean["validation_auc_roc"]},
    {"item": "note", "value": "Validation result is used for model selection and is not final test performance."},
])

validation_summary_path = repo_tables_dir / "distilbert_validation_evaluation_summary.csv"
drive_validation_summary_path = drive_tables_dir / "distilbert_validation_evaluation_summary.csv"

validation_evaluation_summary.to_csv(validation_summary_path, index=False)
validation_evaluation_summary.to_csv(drive_validation_summary_path, index=False)

print("validation evaluation summary saved:", validation_summary_path)
print("validation evaluation summary backup saved:", drive_validation_summary_path)


# 9 final part 17 check
validation_rows_ok = len(validation_predictions_df) == len(validation_df)

validation_metrics_saved = (
    validation_metrics_path.exists()
    and drive_validation_metrics_path.exists()
)

validation_predictions_saved = (
    validation_predictions_path.exists()
    and drive_validation_predictions_path.exists()
)

validation_summary_saved = (
    validation_summary_path.exists()
    and drive_validation_summary_path.exists()
)

required_prediction_columns = [
    "id",
    "text_for_model",
    "true_label",
    "true_label_name",
    "predicted_label",
    "predicted_label_name",
    "probability_SAFE",
    "probability_INJECTION",
    "correct_or_incorrect"
]

validation_prediction_columns_ok = all(
    column in validation_predictions_df.columns
    for column in required_prediction_columns
)

print("final check")
print("validation rows ok:", validation_rows_ok)
print("validation metrics saved:", validation_metrics_saved)
print("validation predictions saved:", validation_predictions_saved)
print("validation summary saved:", validation_summary_saved)
print("validation prediction columns ok:", validation_prediction_columns_ok)

if (
    validation_rows_ok
    and validation_metrics_saved
    and validation_predictions_saved
    and validation_summary_saved
    and validation_prediction_columns_ok
):
    print("part 17 complete")
else:
    print("part 17 needs checking")

#Part 18 - Evaluate DistilBERT on Test Set


In [ ]:
# 1 run prediction on test set
print("running test prediction")

test_prediction_output = trainer.predict(
    test_dataset,
    metric_key_prefix="test"
)

print("test prediction complete")


# 2 collect test outputs
test_logits = test_prediction_output.predictions
test_true_labels = test_prediction_output.label_ids
test_metrics = test_prediction_output.metrics

print("test logits shape:", test_logits.shape)
print("test labels shape:", test_true_labels.shape)
print("test metrics")
print(test_metrics)


# 3 convert logits to probabilities and predictions
test_probabilities = softmax(test_logits)
test_predicted_labels = np.argmax(test_probabilities, axis=1)

test_probability_safe = test_probabilities[:, 0]
test_probability_injection = test_probabilities[:, 1]

print("test probabilities created")
print("test predictions created")


# 4 create test metrics table
test_metrics_clean = {
    "test_loss": test_metrics.get("test_loss"),
    "test_accuracy": test_metrics.get("test_accuracy"),
    "test_precision": test_metrics.get("test_precision"),
    "test_recall": test_metrics.get("test_recall"),
    "test_f1": test_metrics.get("test_f1"),
    "test_auc_roc": test_metrics.get("test_auc_roc"),
    "test_runtime": test_metrics.get("test_runtime"),
    "test_samples_per_second": test_metrics.get("test_samples_per_second"),
    "test_steps_per_second": test_metrics.get("test_steps_per_second"),
}

test_metrics_df = pd.DataFrame([
    {"metric": key, "value": value}
    for key, value in test_metrics_clean.items()
])

print("test metrics table created")
display(test_metrics_df)


# 5 create test prediction table
test_predictions_df = test_df.copy()

test_predictions_df["true_label"] = test_true_labels
test_predictions_df["true_label_name"] = test_predictions_df["true_label"].map(id2label)

test_predictions_df["predicted_label"] = test_predicted_labels
test_predictions_df["predicted_label_name"] = test_predictions_df["predicted_label"].map(id2label)

test_predictions_df["probability_SAFE"] = test_probability_safe
test_predictions_df["probability_INJECTION"] = test_probability_injection

test_predictions_df["correct_or_incorrect"] = np.where(
    test_predictions_df["true_label"] == test_predictions_df["predicted_label"],
    "correct",
    "incorrect"
)

print("test prediction table created")
display(test_predictions_df.head())


# 6 check test prediction counts
test_correct_count = (test_predictions_df["correct_or_incorrect"] == "correct").sum()
test_incorrect_count = (test_predictions_df["correct_or_incorrect"] == "incorrect").sum()

print("test prediction count")
print("correct:", test_correct_count)
print("incorrect:", test_incorrect_count)


# 7 save test metrics and predictions
test_metrics_path = repo_tables_dir / "distilbert_test_metrics.csv"
drive_test_metrics_path = drive_tables_dir / "distilbert_test_metrics.csv"

test_predictions_path = repo_predictions_dir / "distilbert_test_predictions.csv"
drive_test_predictions_path = drive_predictions_dir / "distilbert_test_predictions.csv"

test_metrics_df.to_csv(test_metrics_path, index=False)
test_metrics_df.to_csv(drive_test_metrics_path, index=False)

test_predictions_df.to_csv(test_predictions_path, index=False)
test_predictions_df.to_csv(drive_test_predictions_path, index=False)

print("test metrics saved:", test_metrics_path)
print("test metrics backup saved:", drive_test_metrics_path)

print("test predictions saved:", test_predictions_path)
print("test predictions backup saved:", drive_test_predictions_path)


# 8 save test evaluation summary
test_evaluation_summary = pd.DataFrame([
    {"item": "model_name", "value": model_name},
    {"item": "model_checkpoint", "value": model_checkpoint},
    {"item": "evaluation_split", "value": "test"},
    {"item": "test_rows", "value": len(test_df)},
    {"item": "correct_predictions", "value": test_correct_count},
    {"item": "incorrect_predictions", "value": test_incorrect_count},
    {"item": "accuracy", "value": test_metrics_clean["test_accuracy"]},
    {"item": "precision", "value": test_metrics_clean["test_precision"]},
    {"item": "recall", "value": test_metrics_clean["test_recall"]},
    {"item": "f1", "value": test_metrics_clean["test_f1"]},
    {"item": "auc_roc", "value": test_metrics_clean["test_auc_roc"]},
    {"item": "note", "value": "Test result is final evaluation performance for this DistilBERT run."},
])

test_summary_path = repo_tables_dir / "distilbert_test_evaluation_summary.csv"
drive_test_summary_path = drive_tables_dir / "distilbert_test_evaluation_summary.csv"

test_evaluation_summary.to_csv(test_summary_path, index=False)
test_evaluation_summary.to_csv(drive_test_summary_path, index=False)

print("test evaluation summary saved:", test_summary_path)
print("test evaluation summary backup saved:", drive_test_summary_path)


# 9 final part 18 check
test_rows_ok = len(test_predictions_df) == len(test_df)

test_metrics_saved = (
    test_metrics_path.exists()
    and drive_test_metrics_path.exists()
)

test_predictions_saved = (
    test_predictions_path.exists()
    and drive_test_predictions_path.exists()
)

test_summary_saved = (
    test_summary_path.exists()
    and drive_test_summary_path.exists()
)

required_prediction_columns = [
    "id",
    "text_for_model",
    "true_label",
    "true_label_name",
    "predicted_label",
    "predicted_label_name",
    "probability_SAFE",
    "probability_INJECTION",
    "correct_or_incorrect"
]

test_prediction_columns_ok = all(
    column in test_predictions_df.columns
    for column in required_prediction_columns
)

print("final check")
print("test rows ok:", test_rows_ok)
print("test metrics saved:", test_metrics_saved)
print("test predictions saved:", test_predictions_saved)
print("test summary saved:", test_summary_saved)
print("test prediction columns ok:", test_prediction_columns_ok)

if (
    test_rows_ok
    and test_metrics_saved
    and test_predictions_saved
    and test_summary_saved
    and test_prediction_columns_ok
):
    print("part 18 complete")
else:
    print("part 18 needs checking")

#Part 19 - Generate Confusion Matrix


In [ ]:
# 1 import confusion matrix tools
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

print("confusion matrix tools imported")


# 2 create confusion matrix from test predictions
cm = confusion_matrix(
    test_predictions_df["true_label"],
    test_predictions_df["predicted_label"],
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()

print("confusion matrix created")
print(cm)


# 3 print confusion matrix values
print("confusion matrix values")
print("true negatives:", tn)
print("false positives:", fp)
print("false negatives:", fn)
print("true positives:", tp)


# 4 calculate extra error rates
total_test_rows = len(test_predictions_df)

false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0

true_negative_rate = tn / (tn + fp) if (tn + fp) > 0 else 0
true_positive_rate = tp / (tp + fn) if (tp + fn) > 0 else 0

print("error rate summary")
print("false positive rate:", false_positive_rate)
print("false negative rate:", false_negative_rate)
print("true negative rate:", true_negative_rate)
print("true positive rate:", true_positive_rate)


# 5 create confusion matrix table
confusion_matrix_df = pd.DataFrame([
    {
        "actual_label": 0,
        "actual_label_name": "SAFE",
        "predicted_SAFE": tn,
        "predicted_INJECTION": fp,
    },
    {
        "actual_label": 1,
        "actual_label_name": "INJECTION",
        "predicted_SAFE": fn,
        "predicted_INJECTION": tp,
    },
])

print("confusion matrix table")
display(confusion_matrix_df)


# 6 create confusion matrix summary table
confusion_matrix_summary = pd.DataFrame([
    {"item": "true_negative", "description": "SAFE predicted as SAFE", "value": tn},
    {"item": "false_positive", "description": "SAFE predicted as INJECTION", "value": fp},
    {"item": "false_negative", "description": "INJECTION predicted as SAFE", "value": fn},
    {"item": "true_positive", "description": "INJECTION predicted as INJECTION", "value": tp},
    {"item": "total_test_rows", "description": "Total test prompts", "value": total_test_rows},
    {"item": "false_positive_rate", "description": "SAFE prompts incorrectly flagged as INJECTION", "value": false_positive_rate},
    {"item": "false_negative_rate", "description": "INJECTION prompts incorrectly predicted as SAFE", "value": false_negative_rate},
    {"item": "true_negative_rate", "description": "SAFE prompts correctly predicted as SAFE", "value": true_negative_rate},
    {"item": "true_positive_rate", "description": "INJECTION prompts correctly predicted as INJECTION", "value": true_positive_rate},
])

print("confusion matrix summary")
display(confusion_matrix_summary)


# 7 save confusion matrix tables
confusion_matrix_table_path = repo_tables_dir / "distilbert_confusion_matrix.csv"
drive_confusion_matrix_table_path = drive_tables_dir / "distilbert_confusion_matrix.csv"

confusion_matrix_summary_path = repo_tables_dir / "distilbert_confusion_matrix_summary.csv"
drive_confusion_matrix_summary_path = drive_tables_dir / "distilbert_confusion_matrix_summary.csv"

confusion_matrix_df.to_csv(confusion_matrix_table_path, index=False)
confusion_matrix_df.to_csv(drive_confusion_matrix_table_path, index=False)

confusion_matrix_summary.to_csv(confusion_matrix_summary_path, index=False)
confusion_matrix_summary.to_csv(drive_confusion_matrix_summary_path, index=False)

print("confusion matrix table saved:", confusion_matrix_table_path)
print("confusion matrix table backup saved:", drive_confusion_matrix_table_path)

print("confusion matrix summary saved:", confusion_matrix_summary_path)
print("confusion matrix summary backup saved:", drive_confusion_matrix_summary_path)


# 8 plot confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))

image = ax.imshow(cm)

ax.set_title("DistilBERT Test Confusion Matrix")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])

ax.set_xticklabels(["SAFE", "INJECTION"])
ax.set_yticklabels(["SAFE", "INJECTION"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center"
        )

fig.colorbar(image)
plt.tight_layout()

confusion_matrix_figure_path = repo_figures_dir / "distilbert_confusion_matrix.png"
drive_confusion_matrix_figure_path = drive_figures_dir / "distilbert_confusion_matrix.png"

plt.savefig(confusion_matrix_figure_path, dpi=300, bbox_inches="tight")
plt.savefig(drive_confusion_matrix_figure_path, dpi=300, bbox_inches="tight")

plt.show()

print("confusion matrix figure saved:", confusion_matrix_figure_path)
print("confusion matrix figure backup saved:", drive_confusion_matrix_figure_path)


# 9 final part 19 check
confusion_matrix_values_ok = (tn + fp + fn + tp) == total_test_rows

confusion_matrix_tables_saved = (
    confusion_matrix_table_path.exists()
    and drive_confusion_matrix_table_path.exists()
    and confusion_matrix_summary_path.exists()
    and drive_confusion_matrix_summary_path.exists()
)

confusion_matrix_figures_saved = (
    confusion_matrix_figure_path.exists()
    and drive_confusion_matrix_figure_path.exists()
)

print("final check")
print("confusion matrix values ok:", confusion_matrix_values_ok)
print("confusion matrix tables saved:", confusion_matrix_tables_saved)
print("confusion matrix figures saved:", confusion_matrix_figures_saved)

if confusion_matrix_values_ok and confusion_matrix_tables_saved and confusion_matrix_figures_saved:
    print("part 19 complete")
else:
    print("part 19 needs checking")

#Part 20 - Save  all Predictions

In [ ]:
# 1 define required prediction columns
required_prediction_columns = [
    "id",
    "text_for_model",
    "true_label",
    "true_label_name",
    "predicted_label",
    "predicted_label_name",
    "probability_SAFE",
    "probability_INJECTION",
    "correct_or_incorrect"
]

print("required prediction columns ready")


# 2 create clean validation predictions file
validation_predictions_clean_df = validation_predictions_df[required_prediction_columns].copy()

print("validation predictions prepared")
print("validation rows:", len(validation_predictions_clean_df))
display(validation_predictions_clean_df.head())


# 3 create clean test predictions file
test_predictions_clean_df = test_predictions_df[required_prediction_columns].copy()

print("test predictions prepared")
print("test rows:", len(test_predictions_clean_df))
display(test_predictions_clean_df.head())


# 4 check validation prediction counts
validation_correct_count = (validation_predictions_clean_df["correct_or_incorrect"] == "correct").sum()
validation_incorrect_count = (validation_predictions_clean_df["correct_or_incorrect"] == "incorrect").sum()

print("validation prediction count")
print("correct:", validation_correct_count)
print("incorrect:", validation_incorrect_count)


# 5 check test prediction counts
test_correct_count = (test_predictions_clean_df["correct_or_incorrect"] == "correct").sum()
test_incorrect_count = (test_predictions_clean_df["correct_or_incorrect"] == "incorrect").sum()

print("test prediction count")
print("correct:", test_correct_count)
print("incorrect:", test_incorrect_count)


# 6 save clean prediction files to repo and drive
validation_predictions_path = repo_predictions_dir / "distilbert_validation_predictions.csv"
drive_validation_predictions_path = drive_predictions_dir / "distilbert_validation_predictions.csv"

test_predictions_path = repo_predictions_dir / "distilbert_test_predictions.csv"
drive_test_predictions_path = drive_predictions_dir / "distilbert_test_predictions.csv"

validation_predictions_clean_df.to_csv(validation_predictions_path, index=False)
validation_predictions_clean_df.to_csv(drive_validation_predictions_path, index=False)

test_predictions_clean_df.to_csv(test_predictions_path, index=False)
test_predictions_clean_df.to_csv(drive_test_predictions_path, index=False)

print("validation predictions saved:", validation_predictions_path)
print("validation predictions backup saved:", drive_validation_predictions_path)

print("test predictions saved:", test_predictions_path)
print("test predictions backup saved:", drive_test_predictions_path)


# 7 save prediction summary
prediction_save_summary = pd.DataFrame([
    {
        "split": "validation",
        "rows": len(validation_predictions_clean_df),
        "correct_predictions": validation_correct_count,
        "incorrect_predictions": validation_incorrect_count,
        "repo_path": str(validation_predictions_path),
        "drive_path": str(drive_validation_predictions_path)
    },
    {
        "split": "test",
        "rows": len(test_predictions_clean_df),
        "correct_predictions": test_correct_count,
        "incorrect_predictions": test_incorrect_count,
        "repo_path": str(test_predictions_path),
        "drive_path": str(drive_test_predictions_path)
    }
])

prediction_summary_path = repo_tables_dir / "distilbert_prediction_save_summary.csv"
drive_prediction_summary_path = drive_tables_dir / "distilbert_prediction_save_summary.csv"

prediction_save_summary.to_csv(prediction_summary_path, index=False)
prediction_save_summary.to_csv(drive_prediction_summary_path, index=False)

print("prediction summary saved:", prediction_summary_path)
print("prediction summary backup saved:", drive_prediction_summary_path)

display(prediction_save_summary)


# 8 final part 20 check
validation_predictions_saved = (
    validation_predictions_path.exists()
    and drive_validation_predictions_path.exists()
)

test_predictions_saved = (
    test_predictions_path.exists()
    and drive_test_predictions_path.exists()
)

prediction_summary_saved = (
    prediction_summary_path.exists()
    and drive_prediction_summary_path.exists()
)

validation_columns_ok = all(
    column in validation_predictions_clean_df.columns
    for column in required_prediction_columns
)

test_columns_ok = all(
    column in test_predictions_clean_df.columns
    for column in required_prediction_columns
)

validation_rows_ok = len(validation_predictions_clean_df) == len(validation_df)
test_rows_ok = len(test_predictions_clean_df) == len(test_df)

print("final check")
print("validation predictions saved:", validation_predictions_saved)
print("test predictions saved:", test_predictions_saved)
print("prediction summary saved:", prediction_summary_saved)
print("validation columns ok:", validation_columns_ok)
print("test columns ok:", test_columns_ok)
print("validation rows ok:", validation_rows_ok)
print("test rows ok:", test_rows_ok)

if (
    validation_predictions_saved
    and test_predictions_saved
    and prediction_summary_saved
    and validation_columns_ok
    and test_columns_ok
    and validation_rows_ok
    and test_rows_ok
):
    print("part 20 complete")
else:
    print("part 20 needs checking")

#Part 21 - False Positive and False Negative Analysis

In [ ]:
# 1 identify false positives and false negatives
false_positives_df = test_predictions_df[
    (test_predictions_df["true_label"] == 0) &
    (test_predictions_df["predicted_label"] == 1)
].copy()

false_negatives_df = test_predictions_df[
    (test_predictions_df["true_label"] == 1) &
    (test_predictions_df["predicted_label"] == 0)
].copy()

print("false positive and false negative dataframes created")


# 2 count false positives and false negatives
false_positive_count = len(false_positives_df)
false_negative_count = len(false_negatives_df)

print("error counts")
print("false positives:", false_positive_count)
print("false negatives:", false_negative_count)


# 3 add error type column
false_positives_df["error_type"] = "false_positive"
false_negatives_df["error_type"] = "false_negative"

print("error type columns added")


# 4 choose useful columns for analysis
error_analysis_columns = [
    "id",
    "text_for_model",
    "true_label",
    "true_label_name",
    "predicted_label",
    "predicted_label_name",
    "probability_SAFE",
    "probability_INJECTION",
    "correct_or_incorrect",
    "error_type"
]

false_positives_clean_df = false_positives_df[error_analysis_columns].copy()
false_negatives_clean_df = false_negatives_df[error_analysis_columns].copy()

print("clean error analysis tables created")


# 5 preview false positives
print("false positive examples")
if false_positive_count > 0:
    display(false_positives_clean_df.head())
else:
    print("no false positives found")


# 6 preview false negatives
print("false negative examples")
if false_negative_count > 0:
    display(false_negatives_clean_df.head())
else:
    print("no false negatives found")


# 7 combine all error examples
all_errors_df = pd.concat(
    [false_positives_clean_df, false_negatives_clean_df],
    axis=0,
    ignore_index=True
)

print("all error examples created")
print("all errors:", len(all_errors_df))


# 8 create error summary
error_analysis_summary = pd.DataFrame([
    {
        "error_type": "false_positive",
        "meaning": "SAFE predicted as INJECTION",
        "count": false_positive_count,
        "cybersecurity_risk": "May block or warn on safe user prompts."
    },
    {
        "error_type": "false_negative",
        "meaning": "INJECTION predicted as SAFE",
        "count": false_negative_count,
        "cybersecurity_risk": "More dangerous because malicious prompt-injection attempts may pass through."
    },
    {
        "error_type": "all_errors",
        "meaning": "All incorrect test predictions",
        "count": len(all_errors_df),
        "cybersecurity_risk": "Used for dissertation error analysis."
    }
])

print("error analysis summary")
display(error_analysis_summary)


# 9 save false positive and false negative files
false_positive_path = repo_errors_dir / "distilbert_false_positives.csv"
drive_false_positive_path = drive_errors_dir / "distilbert_false_positives.csv"

false_negative_path = repo_errors_dir / "distilbert_false_negatives.csv"
drive_false_negative_path = drive_errors_dir / "distilbert_false_negatives.csv"

all_errors_path = repo_errors_dir / "distilbert_all_test_errors.csv"
drive_all_errors_path = drive_errors_dir / "distilbert_all_test_errors.csv"

error_summary_path = repo_tables_dir / "distilbert_error_analysis_summary.csv"
drive_error_summary_path = drive_tables_dir / "distilbert_error_analysis_summary.csv"

false_positives_clean_df.to_csv(false_positive_path, index=False)
false_positives_clean_df.to_csv(drive_false_positive_path, index=False)

false_negatives_clean_df.to_csv(false_negative_path, index=False)
false_negatives_clean_df.to_csv(drive_false_negative_path, index=False)

all_errors_df.to_csv(all_errors_path, index=False)
all_errors_df.to_csv(drive_all_errors_path, index=False)

error_analysis_summary.to_csv(error_summary_path, index=False)
error_analysis_summary.to_csv(drive_error_summary_path, index=False)

print("false positives saved:", false_positive_path)
print("false positives backup saved:", drive_false_positive_path)

print("false negatives saved:", false_negative_path)
print("false negatives backup saved:", drive_false_negative_path)

print("all errors saved:", all_errors_path)
print("all errors backup saved:", drive_all_errors_path)

print("error summary saved:", error_summary_path)
print("error summary backup saved:", drive_error_summary_path)


# 10 check expected error counts from confusion matrix
expected_false_positive_count = fp
expected_false_negative_count = fn

false_positive_count_ok = false_positive_count == expected_false_positive_count
false_negative_count_ok = false_negative_count == expected_false_negative_count

print("error count check")
print("false positive count ok:", false_positive_count_ok)
print("false negative count ok:", false_negative_count_ok)


# 11 final part 21 check
error_files_saved = (
    false_positive_path.exists()
    and drive_false_positive_path.exists()
    and false_negative_path.exists()
    and drive_false_negative_path.exists()
    and all_errors_path.exists()
    and drive_all_errors_path.exists()
    and error_summary_path.exists()
    and drive_error_summary_path.exists()
)

all_error_count_ok = len(all_errors_df) == test_incorrect_count

print("final check")
print("error files saved:", error_files_saved)
print("all error count ok:", all_error_count_ok)
print("false positive count ok:", false_positive_count_ok)
print("false negative count ok:", false_negative_count_ok)

if (
    error_files_saved
    and all_error_count_ok
    and false_positive_count_ok
    and false_negative_count_ok
):
    print("part 21 complete")
else:
    print("part 21 needs checking")

#Part 22 - Save Metrics, Tables, and Figures


In [ ]:
# 1 define important output files
important_outputs = [
    {
        "name": "training metrics",
        "repo_path": repo_tables_dir / "distilbert_training_metrics.csv",
        "drive_path": drive_tables_dir / "distilbert_training_metrics.csv",
        "type": "table"
    },
    {
        "name": "training arguments",
        "repo_path": repo_tables_dir / "distilbert_training_arguments_summary.csv",
        "drive_path": drive_tables_dir / "distilbert_training_arguments_summary.csv",
        "type": "table"
    },
    {
        "name": "validation metrics",
        "repo_path": repo_tables_dir / "distilbert_validation_metrics.csv",
        "drive_path": drive_tables_dir / "distilbert_validation_metrics.csv",
        "type": "table"
    },
    {
        "name": "test metrics",
        "repo_path": repo_tables_dir / "distilbert_test_metrics.csv",
        "drive_path": drive_tables_dir / "distilbert_test_metrics.csv",
        "type": "table"
    },
    {
        "name": "confusion matrix table",
        "repo_path": repo_tables_dir / "distilbert_confusion_matrix.csv",
        "drive_path": drive_tables_dir / "distilbert_confusion_matrix.csv",
        "type": "table"
    },
    {
        "name": "confusion matrix summary",
        "repo_path": repo_tables_dir / "distilbert_confusion_matrix_summary.csv",
        "drive_path": drive_tables_dir / "distilbert_confusion_matrix_summary.csv",
        "type": "table"
    },
    {
        "name": "confusion matrix figure",
        "repo_path": repo_figures_dir / "distilbert_confusion_matrix.png",
        "drive_path": drive_figures_dir / "distilbert_confusion_matrix.png",
        "type": "figure"
    },
    {
        "name": "validation predictions",
        "repo_path": repo_predictions_dir / "distilbert_validation_predictions.csv",
        "drive_path": drive_predictions_dir / "distilbert_validation_predictions.csv",
        "type": "prediction"
    },
    {
        "name": "test predictions",
        "repo_path": repo_predictions_dir / "distilbert_test_predictions.csv",
        "drive_path": drive_predictions_dir / "distilbert_test_predictions.csv",
        "type": "prediction"
    },
    {
        "name": "false positives",
        "repo_path": repo_errors_dir / "distilbert_false_positives.csv",
        "drive_path": drive_errors_dir / "distilbert_false_positives.csv",
        "type": "error_analysis"
    },
    {
        "name": "false negatives",
        "repo_path": repo_errors_dir / "distilbert_false_negatives.csv",
        "drive_path": drive_errors_dir / "distilbert_false_negatives.csv",
        "type": "error_analysis"
    },
    {
        "name": "all test errors",
        "repo_path": repo_errors_dir / "distilbert_all_test_errors.csv",
        "drive_path": drive_errors_dir / "distilbert_all_test_errors.csv",
        "type": "error_analysis"
    },
    {
        "name": "error analysis summary",
        "repo_path": repo_tables_dir / "distilbert_error_analysis_summary.csv",
        "drive_path": drive_tables_dir / "distilbert_error_analysis_summary.csv",
        "type": "table"
    },
    {
        "name": "model save summary",
        "repo_path": repo_tables_dir / "distilbert_model_save_summary.csv",
        "drive_path": drive_tables_dir / "distilbert_model_save_summary.csv",
        "type": "table"
    },
    {
        "name": "trainer log history",
        "repo_path": repo_logs_dir / "distilbert_trainer_log_history.csv",
        "drive_path": drive_logs_dir / "distilbert_trainer_log_history.csv",
        "type": "log"
    },
    {
        "name": "trainer state final",
        "repo_path": repo_logs_dir / "distilbert_trainer_state_final.json",
        "drive_path": drive_logs_dir / "distilbert_trainer_state_final.json",
        "type": "log"
    },
    {
        "name": "device information",
        "repo_path": repo_logs_dir / "distilbert_device_info.json",
        "drive_path": drive_logs_dir / "distilbert_device_info.json",
        "type": "log"
    }
]

print("important output list ready")


# 2 copy missing lightweight files if needed
for item in important_outputs:
    repo_path = item["repo_path"]
    drive_path = item["drive_path"]

    if repo_path.exists() and not drive_path.exists():
        drive_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(repo_path, drive_path)
        print("copied repo to drive:", item["name"])

    if drive_path.exists() and not repo_path.exists():
        repo_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, repo_path)
        print("copied drive to repo:", item["name"])

print("file sync check complete")


# 3 create output manifest
manifest_rows = []

for item in important_outputs:
    repo_path = item["repo_path"]
    drive_path = item["drive_path"]

    repo_exists = repo_path.exists()
    drive_exists = drive_path.exists()

    repo_size_kb = round(repo_path.stat().st_size / 1024, 2) if repo_exists else None
    drive_size_kb = round(drive_path.stat().st_size / 1024, 2) if drive_exists else None

    manifest_rows.append({
        "name": item["name"],
        "type": item["type"],
        "repo_exists": repo_exists,
        "drive_exists": drive_exists,
        "repo_size_kb": repo_size_kb,
        "drive_size_kb": drive_size_kb,
        "repo_path": str(repo_path),
        "drive_path": str(drive_path)
    })

distilbert_output_manifest = pd.DataFrame(manifest_rows)

print("output manifest created")
display(distilbert_output_manifest)


# 4 save output manifest
output_manifest_path = repo_tables_dir / "distilbert_saved_outputs_manifest.csv"
drive_output_manifest_path = drive_tables_dir / "distilbert_saved_outputs_manifest.csv"

distilbert_output_manifest.to_csv(output_manifest_path, index=False)
distilbert_output_manifest.to_csv(drive_output_manifest_path, index=False)

print("output manifest saved:", output_manifest_path)
print("output manifest backup saved:", drive_output_manifest_path)


# 5 create final training summary
distilbert_final_training_summary = pd.DataFrame([
    {"item": "model_name", "value": model_name},
    {"item": "model_checkpoint", "value": model_checkpoint},
    {"item": "task_type", "value": task_type},
    {"item": "dataset", "value": "deepset prompt-injection dataset"},
    {"item": "train_rows", "value": len(train_df)},
    {"item": "validation_rows", "value": len(validation_df)},
    {"item": "test_rows", "value": len(test_df)},
    {"item": "max_length", "value": max_length},
    {"item": "epochs", "value": num_train_epochs},
    {"item": "batch_size", "value": per_device_train_batch_size},
    {"item": "learning_rate", "value": learning_rate},
    {"item": "weight_decay", "value": weight_decay},
    {"item": "best_checkpoint", "value": str(best_checkpoint)},
    {"item": "best_validation_f1", "value": best_metric_value},
    {"item": "validation_accuracy", "value": validation_metrics_clean["validation_accuracy"]},
    {"item": "validation_precision", "value": validation_metrics_clean["validation_precision"]},
    {"item": "validation_recall", "value": validation_metrics_clean["validation_recall"]},
    {"item": "validation_f1", "value": validation_metrics_clean["validation_f1"]},
    {"item": "validation_auc_roc", "value": validation_metrics_clean["validation_auc_roc"]},
    {"item": "test_accuracy", "value": test_metrics_clean["test_accuracy"]},
    {"item": "test_precision", "value": test_metrics_clean["test_precision"]},
    {"item": "test_recall", "value": test_metrics_clean["test_recall"]},
    {"item": "test_f1", "value": test_metrics_clean["test_f1"]},
    {"item": "test_auc_roc", "value": test_metrics_clean["test_auc_roc"]},
    {"item": "true_negatives", "value": tn},
    {"item": "false_positives", "value": fp},
    {"item": "false_negatives", "value": fn},
    {"item": "true_positives", "value": tp},
    {"item": "test_correct_predictions", "value": test_correct_count},
    {"item": "test_incorrect_predictions", "value": test_incorrect_count},
    {"item": "model_saved_to_drive", "value": str(best_model_dir)}
])

training_summary_path = repo_tables_dir / "distilbert_training_summary.csv"
drive_training_summary_path = drive_tables_dir / "distilbert_training_summary.csv"

distilbert_final_training_summary.to_csv(training_summary_path, index=False)
distilbert_final_training_summary.to_csv(drive_training_summary_path, index=False)

print("training summary saved:", training_summary_path)
print("training summary backup saved:", drive_training_summary_path)

display(distilbert_final_training_summary)


# 6 final part 22 check
all_repo_outputs_exist = distilbert_output_manifest["repo_exists"].all()
all_drive_outputs_exist = distilbert_output_manifest["drive_exists"].all()

manifest_saved = (
    output_manifest_path.exists()
    and drive_output_manifest_path.exists()
)

training_summary_saved = (
    training_summary_path.exists()
    and drive_training_summary_path.exists()
)

print("final check")
print("all repo outputs exist:", all_repo_outputs_exist)
print("all drive outputs exist:", all_drive_outputs_exist)
print("manifest saved:", manifest_saved)
print("training summary saved:", training_summary_saved)

if all_repo_outputs_exist and all_drive_outputs_exist and manifest_saved and training_summary_saved:
    print("part 22 complete")
else:
    print("part 22 needs checking")

#Part 23 - Save Model and Tokenizer to Google Drive


In [ ]:
# 1 confirm google drive model folder
best_model_dir = drive_model_dir / "best_model"
tokenizer_save_dir = drive_model_dir / "tokenizer"

print("drive model folder:", drive_model_dir)
print("best model folder:", best_model_dir)
print("tokenizer folder:", tokenizer_save_dir)


# 2 check model files
model_config_path = best_model_dir / "config.json"
model_safetensors_path = best_model_dir / "model.safetensors"
pytorch_model_path = best_model_dir / "pytorch_model.bin"
training_args_path = best_model_dir / "training_args.bin"

model_config_exists = model_config_path.exists()
model_safetensors_exists = model_safetensors_path.exists()
pytorch_model_exists = pytorch_model_path.exists()
training_args_exists = training_args_path.exists()

model_weights_exist = model_safetensors_exists or pytorch_model_exists

print("model file check")
print("config.json:", model_config_exists)
print("model.safetensors:", model_safetensors_exists)
print("pytorch_model.bin:", pytorch_model_exists)
print("training_args.bin:", training_args_exists)
print("model weights exist:", model_weights_exist)


# 3 check tokenizer files inside best model folder
best_model_tokenizer_config_path = best_model_dir / "tokenizer_config.json"
best_model_tokenizer_json_path = best_model_dir / "tokenizer.json"
best_model_vocab_path = best_model_dir / "vocab.txt"

best_model_tokenizer_config_exists = best_model_tokenizer_config_path.exists()
best_model_tokenizer_json_exists = best_model_tokenizer_json_path.exists()
best_model_vocab_exists = best_model_vocab_path.exists()

best_model_tokenizer_ok = (
    best_model_tokenizer_config_exists
    and (best_model_tokenizer_json_exists or best_model_vocab_exists)
)

print("tokenizer files in best model folder")
print("tokenizer_config.json:", best_model_tokenizer_config_exists)
print("tokenizer.json:", best_model_tokenizer_json_exists)
print("vocab.txt:", best_model_vocab_exists)
print("tokenizer ok:", best_model_tokenizer_ok)


# 4 check separate tokenizer folder
tokenizer_config_path = tokenizer_save_dir / "tokenizer_config.json"
tokenizer_json_path = tokenizer_save_dir / "tokenizer.json"
vocab_path = tokenizer_save_dir / "vocab.txt"

tokenizer_config_exists = tokenizer_config_path.exists()
tokenizer_json_exists = tokenizer_json_path.exists()
vocab_exists = vocab_path.exists()

separate_tokenizer_ok = (
    tokenizer_config_exists
    and (tokenizer_json_exists or vocab_exists)
)

print("separate tokenizer folder check")
print("tokenizer_config.json:", tokenizer_config_exists)
print("tokenizer.json:", tokenizer_json_exists)
print("vocab.txt:", vocab_exists)
print("separate tokenizer ok:", separate_tokenizer_ok)


# 5 list saved files
best_model_files = sorted([file.name for file in best_model_dir.iterdir() if file.is_file()])
separate_tokenizer_files = sorted([file.name for file in tokenizer_save_dir.iterdir() if file.is_file()])

print("best model files")
print(best_model_files)

print("separate tokenizer files")
print(separate_tokenizer_files)


# 6 calculate folder sizes
def get_folder_size_mb(folder_path):
    total_size = 0

    for path in folder_path.rglob("*"):
        if path.is_file():
            total_size += path.stat().st_size

    return round(total_size / (1024 * 1024), 2)


best_model_size_mb = get_folder_size_mb(best_model_dir)
tokenizer_size_mb = get_folder_size_mb(tokenizer_save_dir)

print("folder size check")
print("best model size mb:", best_model_size_mb)
print("tokenizer size mb:", tokenizer_size_mb)


# 7 save drive model backup summary
drive_model_backup_summary = pd.DataFrame([
    {"item": "model_name", "value": model_name},
    {"item": "model_checkpoint", "value": model_checkpoint},
    {"item": "best_checkpoint", "value": str(best_checkpoint)},
    {"item": "best_validation_f1", "value": best_metric_value},
    {"item": "drive_model_dir", "value": str(drive_model_dir)},
    {"item": "best_model_dir", "value": str(best_model_dir)},
    {"item": "tokenizer_save_dir", "value": str(tokenizer_save_dir)},
    {"item": "config_json_exists", "value": model_config_exists},
    {"item": "model_safetensors_exists", "value": model_safetensors_exists},
    {"item": "pytorch_model_bin_exists", "value": pytorch_model_exists},
    {"item": "model_weights_exist", "value": model_weights_exist},
    {"item": "training_args_bin_exists", "value": training_args_exists},
    {"item": "best_model_tokenizer_ok", "value": best_model_tokenizer_ok},
    {"item": "separate_tokenizer_ok", "value": separate_tokenizer_ok},
    {"item": "best_model_size_mb", "value": best_model_size_mb},
    {"item": "tokenizer_size_mb", "value": tokenizer_size_mb},
    {"item": "best_model_files", "value": ", ".join(best_model_files)},
    {"item": "separate_tokenizer_files", "value": ", ".join(separate_tokenizer_files)},
    {"item": "github_commit_note", "value": "Large fine-tuned model files are saved in Google Drive and should not be committed to GitHub."}
])

drive_model_backup_summary_path = drive_tables_dir / "distilbert_drive_model_backup_summary.csv"
repo_model_backup_summary_path = repo_tables_dir / "distilbert_drive_model_backup_summary.csv"

drive_model_backup_summary.to_csv(drive_model_backup_summary_path, index=False)
drive_model_backup_summary.to_csv(repo_model_backup_summary_path, index=False)

print("drive model backup summary saved:", drive_model_backup_summary_path)
print("repo model backup summary saved:", repo_model_backup_summary_path)

display(drive_model_backup_summary)


# 8 final part 23 check
model_backup_ok = (
    model_config_exists
    and model_weights_exist
    and best_model_tokenizer_ok
    and separate_tokenizer_ok
)

summary_saved = (
    drive_model_backup_summary_path.exists()
    and repo_model_backup_summary_path.exists()
)

print("final check")
print("model backup ok:", model_backup_ok)
print("summary saved:", summary_saved)

if model_backup_ok and summary_saved:
    print("part 23 complete")
else:
    print("part 23 needs checking")